In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# File paths
# =========================================================
bond_file = "yclv3aucasl1v19k.csv"
company_file = "CapIQcompanylistWcusips.xlsx"

# =========================================================
# Parameters
# =========================================================
screen_date = pd.Timestamp("2024-01-01")
min_remaining_years = 3
max_remaining_years = 10

output_folder = "bond_screen_outputs_2024_2025"
zip_filename = "bond_screen_outputs_2024_2025.zip"

# =========================================================
# Create output folder
# =========================================================
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# Load raw bond list
# =========================================================
df = pd.read_csv(bond_file, low_memory=False)

# =========================================================
# Standardize key columns
# =========================================================
df["STDT"] = pd.to_datetime(df["STDT"], errors="coerce")
df["mtrty_dt"] = pd.to_datetime(df["mtrty_dt"], errors="coerce")

df["cusip_id"] = df["cusip_id"].astype(str).str.strip().str.upper()
df["COMPANY_SYMBOL"] = df["COMPANY_SYMBOL"].astype(str).str.strip().str.upper()
df["issuer_nm"] = df["issuer_nm"].astype(str).str.strip()
df["grade"] = df["grade"].astype(str).str.strip().str.upper()

# Drop rows missing essential fields
df = df.dropna(subset=["cusip_id", "COMPANY_SYMBOL", "STDT", "mtrty_dt", "grade"]).copy()

# =========================================================
# Keep only investment-grade bonds
# =========================================================
df = df[df["grade"] == "I"].copy()

# =========================================================
# First TRACE appearance per CUSIP
# =========================================================
cusip_first_trace = (
    df.groupby("cusip_id", as_index=False)["STDT"]
    .min()
    .rename(columns={"STDT": "first_trace_date"})
)

df = df.merge(cusip_first_trace, on="cusip_id", how="left")

# =========================================================
# Compute maturity measures
# =========================================================
# 1. Bond maturity proxy = maturity date - first TRACE appearance
df["years_to_maturity_from_first_trace"] = (
    (df["mtrty_dt"] - df["first_trace_date"]).dt.days / 365.25
)

# 2. Remaining maturity at the start of the sample
df["years_to_maturity_asof_2024_01_01"] = (
    (df["mtrty_dt"] - screen_date).dt.days / 365.25
)

# =========================================================
# HARD FILTER:
# Remove every bond that does NOT meet the constraint
# =========================================================
eligible_mask = (
    (df["years_to_maturity_asof_2024_01_01"] >= min_remaining_years) &
    (df["years_to_maturity_asof_2024_01_01"] <= max_remaining_years)
)

df = df.loc[eligible_mask].copy()

# =========================================================
# Load and clean company-sector map
# IMPORTANT: use Sheet1
# =========================================================
company_raw = pd.read_excel(company_file, sheet_name="Sheet1", header=None)

# Actual company observations begin at row 5
company_map = company_raw.iloc[5:, [0, 1, 4]].copy()
company_map.columns = ["entity_name", "Ticker", "Sector"]

company_map["entity_name"] = company_map["entity_name"].astype(str).str.strip()
company_map["Ticker"] = company_map["Ticker"].astype(str).str.strip().str.upper()
company_map["Sector"] = company_map["Sector"].astype(str).str.strip()

# Remove blank/junk rows
company_map = company_map[
    company_map["Ticker"].notna() &
    (company_map["Ticker"] != "") &
    (company_map["Ticker"] != "NAN")
].copy()

company_map = company_map[
    company_map["Sector"].notna() &
    (company_map["Sector"] != "") &
    (company_map["Sector"] != "NAN")
].copy()

# One row per ticker
company_map = company_map.drop_duplicates(subset=["Ticker"])

# =========================================================
# Merge sector onto eligible bonds only
# =========================================================
df = df.merge(
    company_map[["Ticker", "Sector", "entity_name"]],
    left_on="COMPANY_SYMBOL",
    right_on="Ticker",
    how="left"
)

df = df.drop(columns=["Ticker"], errors="ignore")

# =========================================================
# Remove exact duplicate rows
# =========================================================
df = df.drop_duplicates().copy()

# =========================================================
# Collapse to one row per CUSIP
# Only eligible bonds survive to this file
# =========================================================
eligible_bonds = (
    df.sort_values(["cusip_id", "first_trace_date"])
    .groupby("cusip_id", as_index=False)
    .agg(
        COMPANY_SYMBOL=("COMPANY_SYMBOL", "first"),
        issuer_nm=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        Sector=("Sector", "first"),
        grade=("grade", "first"),
        first_trace_date=("first_trace_date", "first"),
        mtrty_dt=("mtrty_dt", "first"),
        years_to_maturity_from_first_trace=("years_to_maturity_from_first_trace", "first"),
        years_to_maturity_asof_2024_01_01=("years_to_maturity_asof_2024_01_01", "first")
    )
    .sort_values(["COMPANY_SYMBOL", "cusip_id"])
    .reset_index(drop=True)
)

# Extra safety check
eligible_bonds = eligible_bonds[
    (eligible_bonds["years_to_maturity_asof_2024_01_01"] >= min_remaining_years) &
    (eligible_bonds["years_to_maturity_asof_2024_01_01"] <= max_remaining_years)
].copy()

# =========================================================
# WRDS pull list
# =========================================================
wrds_cusip_list = (
    eligible_bonds[["cusip_id"]]
    .drop_duplicates()
    .sort_values("cusip_id")
    .reset_index(drop=True)
)

# =========================================================
# Company summary
# =========================================================
company_summary = (
    eligible_bonds.groupby(["COMPANY_SYMBOL", "Sector"], dropna=False)
    .agg(
        issuer_name=("issuer_nm", "first"),
        n_unique_cusips=("cusip_id", "nunique")
    )
    .reset_index()
    .sort_values(["n_unique_cusips", "COMPANY_SYMBOL"], ascending=[False, True])
)

# =========================================================
# Company-to-CUSIP mapping
# =========================================================
company_to_cusips = (
    eligible_bonds[
        [
            "COMPANY_SYMBOL",
            "issuer_nm",
            "entity_name",
            "Sector",
            "cusip_id",
            "first_trace_date",
            "mtrty_dt",
            "years_to_maturity_from_first_trace",
            "years_to_maturity_asof_2024_01_01"
        ]
    ]
    .drop_duplicates()
    .sort_values(["COMPANY_SYMBOL", "cusip_id"])
    .reset_index(drop=True)
)

# =========================================================
# Diagnostics
# =========================================================
missing_sector = eligible_bonds[eligible_bonds["Sector"].isna()].copy()

diagnostics = pd.DataFrame({
    "metric": [
        "unique_companies",
        "unique_cusips",
        "cusips_missing_sector",
        "min_remaining_maturity",
        "max_remaining_maturity"
    ],
    "value": [
        eligible_bonds["COMPANY_SYMBOL"].nunique(),
        eligible_bonds["cusip_id"].nunique(),
        eligible_bonds["Sector"].isna().sum(),
        eligible_bonds["years_to_maturity_asof_2024_01_01"].min(),
        eligible_bonds["years_to_maturity_asof_2024_01_01"].max()
    ]
})

# =========================================================
# Save outputs
# =========================================================
eligible_bonds_file = os.path.join(output_folder, "eligible_ig_bonds_3_to_10y_remaining_asof_2024_01_01.csv")
wrds_cusips_file = os.path.join(output_folder, "wrds_pull_cusip_list_ig_3_to_10y_remaining_asof_2024_01_01.csv")
wrds_cusips_txt_file = os.path.join(output_folder, "wrds_pull_cusip_list_ig_3_to_10y_remaining_asof_2024_01_01.txt")
company_summary_file = os.path.join(output_folder, "company_summary_ig_3_to_10y_remaining_asof_2024_01_01.csv")
company_to_cusips_file = os.path.join(output_folder, "company_to_cusips_ig_3_to_10y_remaining_asof_2024_01_01.csv")
diagnostics_file = os.path.join(output_folder, "bond_screen_match_diagnostics.csv")
missing_sector_file = os.path.join(output_folder, "companies_missing_sector_after_merge.csv")

eligible_bonds.to_csv(eligible_bonds_file, index=False)
wrds_cusip_list.to_csv(wrds_cusips_file, index=False)
wrds_cusip_list.to_csv(wrds_cusips_txt_file, index=False, header=False)
company_summary.to_csv(company_summary_file, index=False)
company_to_cusips.to_csv(company_to_cusips_file, index=False)
diagnostics.to_csv(diagnostics_file, index=False)
missing_sector.to_csv(missing_sector_file, index=False)

# =========================================================
# Create zip file with all outputs
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# Print checks
# =========================================================
print("=== Bond Screen Complete ===")
print("Only bonds satisfying the maturity constraint were kept.")
print("Screen date:", screen_date.date())
print("Remaining maturity filter:", f"{min_remaining_years} to {max_remaining_years} years inclusive")
print()
print("Unique companies:", eligible_bonds["COMPANY_SYMBOL"].nunique())
print("Unique CUSIPs:", eligible_bonds["cusip_id"].nunique())
print("CUSIPs missing sector:", eligible_bonds["Sector"].isna().sum())
print("Min remaining maturity kept:", eligible_bonds["years_to_maturity_asof_2024_01_01"].min())
print("Max remaining maturity kept:", eligible_bonds["years_to_maturity_asof_2024_01_01"].max())
print()
print("Zip file created:", zip_filename)

# Optional Colab auto-download
try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== Bond Screen Complete ===
Only bonds satisfying the maturity constraint were kept.
Screen date: 2024-01-01
Remaining maturity filter: 3 to 10 years inclusive

Unique companies: 248
Unique CUSIPs: 12968
CUSIPs missing sector: 0
Min remaining maturity kept: 3.0006844626967832
Max remaining maturity kept: 9.998631074606434

Zip file created: bond_screen_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# FILE PATHS
# =========================================================
quarter_files = [
    "2024Q1.csv",
    "2024Q2.csv",
    "2024Q3.csv",
    "2024Q4.csv",
    "2025Q1.csv",
    "2025Q2.csv",
    "2025Q3.csv",
    "2025Q4.csv",
]

eligible_bonds_file = "eligible_ig_bonds_3_to_10y_remaining_asof_2024_01_01.csv"
treasury_2024_file = "2024Treasury.csv"
treasury_2025_file = "2025Treasury.csv"

output_folder = "trace_clean_outputs_2024_2025"
zip_filename = "trace_clean_outputs_2024_2025.zip"

# =========================================================
# CREATE OUTPUT FOLDER
# =========================================================
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def clean_numeric(series):
    """
    Remove commas, dollar signs, percent signs, whitespace, and coerce to numeric.
    """
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def signed_yield(yld_pt, yld_sign_cd):
    """
    Convert yield plus sign code into a signed yield.
    If sign code indicates negative, flip sign.
    Otherwise keep the numeric value as-is.
    """
    y = clean_numeric(yld_pt)

    sign = yld_sign_cd.astype(str).str.strip().str.upper()
    neg_mask = sign.isin(["-", "N", "NEG", "NEGATIVE"])

    y = np.where(neg_mask, -np.abs(y), y)
    return pd.Series(y, index=yld_pt.index)

def clean_treasury_file(path):
    """
    Standardize a Treasury file into a daily curve format.
    """
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    if "Date" not in df.columns:
        raise ValueError(f"'Date' column not found in {path}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).copy()

    # Convert all non-date columns to numeric
    tenor_cols = [c for c in df.columns if c != "Date"]
    for col in tenor_cols:
        df[col] = clean_numeric(df[col])

    df = df.sort_values("Date").drop_duplicates(subset=["Date"]).reset_index(drop=True)
    return df

def weighted_avg(group, value_col, weight_col):
    vals = group[value_col]
    wts = group[weight_col]

    mask = vals.notna() & wts.notna() & (wts > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(vals[mask], weights=wts[mask])

# =========================================================
# LOAD ELIGIBLE BOND METADATA
# This comes from your earlier CUSIP screening step.
# It already restricts to IG + remaining maturity 3-10y as of 2024-01-01.
# =========================================================
meta = pd.read_csv(eligible_bonds_file, low_memory=False)

meta["cusip_id"] = meta["cusip_id"].astype(str).str.strip().str.upper()
meta["COMPANY_SYMBOL"] = meta["COMPANY_SYMBOL"].astype(str).str.strip().str.upper()
meta["issuer_nm"] = meta["issuer_nm"].astype(str).str.strip()
meta["entity_name"] = meta["entity_name"].astype(str).str.strip()
meta["Sector"] = meta["Sector"].astype(str).str.strip()
meta["grade"] = meta["grade"].astype(str).str.strip().str.upper()

meta["first_trace_date"] = pd.to_datetime(meta["first_trace_date"], errors="coerce")
meta["mtrty_dt"] = pd.to_datetime(meta["mtrty_dt"], errors="coerce")

meta["years_to_maturity_from_first_trace"] = pd.to_numeric(
    meta["years_to_maturity_from_first_trace"], errors="coerce"
)
meta["years_to_maturity_asof_2024_01_01"] = pd.to_numeric(
    meta["years_to_maturity_asof_2024_01_01"], errors="coerce"
)

# Safety check: only IG bonds should remain
meta = meta[meta["grade"] == "I"].copy()

# One row per CUSIP
meta = meta.drop_duplicates(subset=["cusip_id"]).copy()

# =========================================================
# LOAD + STANDARDIZE TREASURY FILES
# Not yet matched to bonds at this stage, but standardized and saved now
# for later spread construction.
# =========================================================
treasury_frames = []

if os.path.exists(treasury_2024_file):
    treasury_frames.append(clean_treasury_file(treasury_2024_file))
else:
    print(f"WARNING: {treasury_2024_file} not found")

if os.path.exists(treasury_2025_file):
    treasury_frames.append(clean_treasury_file(treasury_2025_file))
else:
    print(f"WARNING: {treasury_2025_file} not found")

if len(treasury_frames) == 0:
    raise FileNotFoundError("No Treasury files found.")

treasury_all = (
    pd.concat(treasury_frames, ignore_index=True)
    .sort_values("Date")
    .drop_duplicates(subset=["Date"])
    .reset_index(drop=True)
)

treasury_output_file = os.path.join(output_folder, "combined_treasury_curve_2024_2025.csv")
treasury_all.to_csv(treasury_output_file, index=False)

# =========================================================
# PROCESS EACH QUARTER
# =========================================================
quarter_panel_files = []
quarter_trade_files = []
quarter_summary_rows = []

for qfile in quarter_files:
    if not os.path.exists(qfile):
        print(f"Skipping missing file: {qfile}")
        continue

    print(f"\nProcessing {qfile} ...")
    df = pd.read_csv(qfile, low_memory=False)

    # ---------------------------------------------
    # Standardize columns if present
    # ---------------------------------------------
    df.columns = [c.strip() for c in df.columns]

    # Key fields expected from WRDS TRACE pull
    expected_cols = [
        "cusip_id",
        "bond_sym_id",
        "company_symbol",
        "issuer",
        "trd_exctn_dt",
        "trd_exctn_tm",
        "trans_dt",
        "msg_seq_nb",
        "trc_st",
        "ascii_rptd_vol_tx",
        "rptd_pr",
        "yld_sign_cd",
        "yld_pt",
        "asof_cd",
        "sale_cndtn_cd",
        "sale_cndtn2_cd",
        "spcl_trd_fl",
        "diss_rptg_side_cd",
        "side",
        "chng_cd",
        "sttl_dt",
        "rptg_party_type",
        "contra_party_type",
        "ats_indicator",
        "orig_msg_seq_nb",
        "orig_trd_exctn_dt",
        "orig_trd_exctn_tm",
        "orig_rptd_pr",
        "orig_yld_sign_cd",
        "orig_yld_pt",
        "orig_ascii_rptd_vol_tx",
        "orig_side",
        "orig_asof_cd",
        "orig_sale_cndtn_cd",
        "orig_sale_cndtn2_cd",
        "orig_sttl_dt",
        "orig_rptg_party_type",
        "orig_contra_party_type",
        "orig_ats_indicator",
    ]

    missing_expected = [c for c in ["cusip_id", "trd_exctn_dt", "rptd_pr", "yld_pt"] if c not in df.columns]
    if len(missing_expected) > 0:
        raise ValueError(f"{qfile} is missing required columns: {missing_expected}")

    # ---------------------------------------------
    # Basic standardization
    # ---------------------------------------------
    df["cusip_id"] = df["cusip_id"].astype(str).str.strip().str.upper()

    if "company_symbol" in df.columns:
        df["company_symbol"] = df["company_symbol"].astype(str).str.strip().str.upper()
    else:
        df["company_symbol"] = np.nan

    if "issuer" in df.columns:
        df["issuer"] = df["issuer"].astype(str).str.strip()
    else:
        df["issuer"] = np.nan

    df["trd_exctn_dt"] = pd.to_datetime(df["trd_exctn_dt"], errors="coerce")
    if "trans_dt" in df.columns:
        df["trans_dt"] = pd.to_datetime(df["trans_dt"], errors="coerce")
    if "sttl_dt" in df.columns:
        df["sttl_dt"] = pd.to_datetime(df["sttl_dt"], errors="coerce")
    if "orig_trd_exctn_dt" in df.columns:
        df["orig_trd_exctn_dt"] = pd.to_datetime(df["orig_trd_exctn_dt"], errors="coerce")
    if "orig_sttl_dt" in df.columns:
        df["orig_sttl_dt"] = pd.to_datetime(df["orig_sttl_dt"], errors="coerce")

    df["rptd_pr"] = clean_numeric(df["rptd_pr"])
    df["ascii_rptd_vol_tx"] = clean_numeric(df["ascii_rptd_vol_tx"]) if "ascii_rptd_vol_tx" in df.columns else np.nan
    df["yld_clean"] = signed_yield(df["yld_pt"], df["yld_sign_cd"]) if "yld_sign_cd" in df.columns else clean_numeric(df["yld_pt"])

    if "orig_rptd_pr" in df.columns:
        df["orig_rptd_pr"] = clean_numeric(df["orig_rptd_pr"])
    if "orig_yld_pt" in df.columns:
        if "orig_yld_sign_cd" in df.columns:
            df["orig_yld_clean"] = signed_yield(df["orig_yld_pt"], df["orig_yld_sign_cd"])
        else:
            df["orig_yld_clean"] = clean_numeric(df["orig_yld_pt"])
    if "orig_ascii_rptd_vol_tx" in df.columns:
        df["orig_ascii_rptd_vol_tx"] = clean_numeric(df["orig_ascii_rptd_vol_tx"])

    # ---------------------------------------------
    # Drop rows missing essential fields
    # ---------------------------------------------
    df = df.dropna(subset=["cusip_id", "trd_exctn_dt", "rptd_pr", "yld_clean"]).copy()

    # ---------------------------------------------
    # Keep only trades for eligible IG CUSIPs
    # ---------------------------------------------
    df = df.merge(
        meta,
        on="cusip_id",
        how="inner",
        suffixes=("", "_meta")
    )

    # ---------------------------------------------
    # Clean obvious bad numeric values
    # Keep this close to your original notebook logic.
    # ---------------------------------------------
    df = df[np.isfinite(df["rptd_pr"]) & np.isfinite(df["yld_clean"])].copy()
    df = df[(df["rptd_pr"] > 0) & (df["rptd_pr"] <= 300)].copy()
    df = df[(df["yld_clean"] >= -100) & (df["yld_clean"] <= 100)].copy()

    # ---------------------------------------------
    # Remove exact duplicate rows
    # ---------------------------------------------
    df = df.drop_duplicates().copy()

    # ---------------------------------------------
    # Save cleaned trade-level quarter file
    # ---------------------------------------------
    qlabel = os.path.splitext(os.path.basename(qfile))[0]
    quarter_trade_file = os.path.join(output_folder, f"{qlabel}_cleaned_trades.csv.gz")
    df.to_csv(quarter_trade_file, index=False, compression="gzip")
    quarter_trade_files.append(quarter_trade_file)

    # ---------------------------------------------
    # Build one-row-per-CUSIP-day panel for this quarter
    # ---------------------------------------------
    group_keys = ["cusip_id", "trd_exctn_dt"]

    panel = (
        df.groupby(group_keys, dropna=False)
        .agg(
            company_symbol=("COMPANY_SYMBOL", "first"),
            company_symbol_trace=("company_symbol", "first"),
            issuer_nm=("issuer_nm", "first"),
            entity_name=("entity_name", "first"),
            sector=("Sector", "first"),
            grade=("grade", "first"),
            first_trace_date=("first_trace_date", "first"),
            mtrty_dt=("mtrty_dt", "first"),
            years_to_maturity_from_first_trace=("years_to_maturity_from_first_trace", "first"),
            years_to_maturity_asof_2024_01_01=("years_to_maturity_asof_2024_01_01", "first"),
            median_price=("rptd_pr", "median"),
            mean_price=("rptd_pr", "mean"),
            median_yield=("yld_clean", "median"),
            mean_yield=("yld_clean", "mean"),
            n_trades=("rptd_pr", "size"),
            total_quantity=("ascii_rptd_vol_tx", "sum"),
        )
        .reset_index()
        .rename(columns={"trd_exctn_dt": "trade_date"})
    )

    # Volume-weighted fields
    weighted = (
        df.groupby(group_keys)
        .apply(lambda g: pd.Series({
            "vw_price": weighted_avg(g, "rptd_pr", "ascii_rptd_vol_tx"),
            "vw_yield": weighted_avg(g, "yld_clean", "ascii_rptd_vol_tx"),
        }))
        .reset_index()
        .rename(columns={"trd_exctn_dt": "trade_date"})
    )

    panel = panel.merge(weighted, on=["cusip_id", "trade_date"], how="left")

    quarter_panel_file = os.path.join(output_folder, f"{qlabel}_cusip_day_panel.csv")
    panel.to_csv(quarter_panel_file, index=False)
    quarter_panel_files.append(quarter_panel_file)

    # Quarter summary
    quarter_summary_rows.append({
        "quarter_file": qfile,
        "cleaned_trade_rows": len(df),
        "cusip_day_rows": len(panel),
        "unique_cusips": panel["cusip_id"].nunique(),
        "unique_companies": panel["company_symbol"].nunique(),
        "start_date": panel["trade_date"].min(),
        "end_date": panel["trade_date"].max(),
    })

    print(f"Finished {qfile}: {len(df):,} cleaned trades, {len(panel):,} CUSIP-day rows")

# =========================================================
# CHECK THAT AT LEAST ONE QUARTER PROCESSED
# =========================================================
if len(quarter_panel_files) == 0:
    raise RuntimeError("No quarter files were processed. Check file names and locations.")

# =========================================================
# COMBINE ALL QUARTER CUSIP-DAY PANELS
# Because quarters do not overlap, concatenation is valid.
# =========================================================
all_panels = [pd.read_csv(f, low_memory=False, parse_dates=["trade_date", "first_trace_date", "mtrty_dt"]) for f in quarter_panel_files]
cusip_day_panel = pd.concat(all_panels, ignore_index=True)

cusip_day_panel = (
    cusip_day_panel
    .sort_values(["trade_date", "cusip_id"])
    .reset_index(drop=True)
)

cusip_day_panel_file = os.path.join(output_folder, "cusip_day_panel_cleaned.csv")
cusip_day_panel.to_csv(cusip_day_panel_file, index=False)

# =========================================================
# COVERAGE SUMMARY
# =========================================================
coverage_summary = (
    cusip_day_panel.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        sector=("sector", "first"),
        grade=("grade", "first"),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
        n_trade_days=("trade_date", "nunique"),
        total_trades=("n_trades", "sum"),
        avg_trades_per_day=("n_trades", "mean"),
        median_price_mean=("median_price", "mean"),
        median_yield_mean=("median_yield", "mean"),
        median_price_std=("median_price", "std"),
        median_yield_std=("median_yield", "std"),
        total_quantity=("total_quantity", "sum"),
        years_to_maturity_from_first_trace=("years_to_maturity_from_first_trace", "first"),
        years_to_maturity_asof_2024_01_01=("years_to_maturity_asof_2024_01_01", "first"),
        maturity_date=("mtrty_dt", "first"),
    )
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)

coverage_summary_file = os.path.join(output_folder, "cusip_day_coverage_summary.csv")
coverage_summary.to_csv(coverage_summary_file, index=False)

# =========================================================
# WIDE DAILY YIELD PANEL
# =========================================================
yield_panel_wide = cusip_day_panel.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="median_yield",
    aggfunc="first"
).sort_index()

yield_panel_wide_file = os.path.join(output_folder, "cusip_day_yield_panel_wide.csv")
yield_panel_wide.to_csv(yield_panel_wide_file)

# =========================================================
# QUARTER SUMMARY REPORT
# =========================================================
quarter_summary_df = pd.DataFrame(quarter_summary_rows)
quarter_summary_file = os.path.join(output_folder, "quarter_processing_summary.csv")
quarter_summary_df.to_csv(quarter_summary_file, index=False)

# =========================================================
# MANIFEST
# =========================================================
manifest = pd.DataFrame({
    "output_file": sorted(os.listdir(output_folder))
})
manifest_file = os.path.join(output_folder, "output_manifest.csv")
manifest.to_csv(manifest_file, index=False)

# =========================================================
# CREATE ZIP FILE
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("\n=== TRACE CLEANING COMPLETE ===")
print(f"Processed quarter files: {len(quarter_panel_files)}")
print(f"Combined CUSIP-day rows: {len(cusip_day_panel):,}")
print(f"Unique CUSIPs: {cusip_day_panel['cusip_id'].nunique():,}")
print(f"Unique companies: {cusip_day_panel['company_symbol'].nunique():,}")
print(f"Date range: {cusip_day_panel['trade_date'].min()} to {cusip_day_panel['trade_date'].max()}")
print(f"Treasury curve rows: {len(treasury_all):,}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

# Optional Colab auto-download
try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")


Processing 2024Q1.csv ...


/tmp/ipykernel_1507/2290908115.py:326: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


Finished 2024Q1.csv: 1,677,770 cleaned trades, 106,970 CUSIP-day rows

Processing 2024Q2.csv ...


/tmp/ipykernel_1507/2290908115.py:326: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


Finished 2024Q2.csv: 1,653,305 cleaned trades, 116,674 CUSIP-day rows

Processing 2024Q3.csv ...


...

Finished 2025Q3.csv: 1,902,276 cleaned trades, 137,068 CUSIP-day rows

Processing 2025Q4.csv ...


/tmp/ipykernel_1507/2290908115.py:326: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


Finished 2025Q4.csv: 2,003,243 cleaned trades, 136,110 CUSIP-day rows

=== TRACE CLEANING COMPLETE ===
Processed quarter files: 8
Combined CUSIP-day rows: 991,938
Unique CUSIPs: 4,495
Unique companies: 233
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Treasury curve rows: 499
Output folder created: trace_clean_outputs_2024_2025
Zip file created: trace_clean_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from itertools import combinations

# =========================================================
# FILE PATHS
# =========================================================
panel_file = "cusip_day_panel_cleaned.csv"
coverage_file = "cusip_day_coverage_summary.csv"

output_folder = "usable_bonds_outputs_2024_2025"
zip_filename = "usable_bonds_outputs_2024_2025.zip"

# =========================================================
# THRESHOLDS
# Keep these easy to change
# =========================================================
min_trade_days = 100
min_avg_trades_per_day = 3
min_yield_std = 0.02
min_price_std = 0.02

# =========================================================
# CREATE OUTPUT FOLDER
# =========================================================
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# LOAD INPUTS
# =========================================================
panel = pd.read_csv(panel_file, low_memory=False)
coverage = pd.read_csv(coverage_file, low_memory=False)

# =========================================================
# STANDARDIZE TYPES
# =========================================================
if "trade_date" in panel.columns:
    panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
if "first_trace_date" in panel.columns:
    panel["first_trace_date"] = pd.to_datetime(panel["first_trace_date"], errors="coerce")
if "mtrty_dt" in panel.columns:
    panel["mtrty_dt"] = pd.to_datetime(panel["mtrty_dt"], errors="coerce")

panel["cusip_id"] = panel["cusip_id"].astype(str).str.strip().str.upper()
coverage["cusip_id"] = coverage["cusip_id"].astype(str).str.strip().str.upper()

numeric_cols_panel = [
    "median_price", "mean_price", "median_yield", "mean_yield",
    "n_trades", "total_quantity", "vw_price", "vw_yield",
    "years_to_maturity_from_first_trace", "years_to_maturity_asof_2024_01_01"
]
for col in numeric_cols_panel:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

numeric_cols_coverage = [
    "n_trade_days", "total_trades", "avg_trades_per_day",
    "median_price_mean", "median_yield_mean",
    "median_price_std", "median_yield_std",
    "total_quantity", "years_to_maturity_from_first_trace",
    "years_to_maturity_asof_2024_01_01"
]
for col in numeric_cols_coverage:
    if col in coverage.columns:
        coverage[col] = pd.to_numeric(coverage[col], errors="coerce")

# =========================================================
# APPLY USABLE-BONDS FILTER
# =========================================================
usable_mask = (
    (coverage["n_trade_days"] >= min_trade_days) &
    (coverage["avg_trades_per_day"] >= min_avg_trades_per_day) &
    (coverage["median_yield_std"] >= min_yield_std) &
    (coverage["median_price_std"] >= min_price_std)
)

usable_bonds = coverage.loc[usable_mask].copy()
dropped_bonds = coverage.loc[~usable_mask].copy()

usable_cusips = set(usable_bonds["cusip_id"])

# =========================================================
# FILTER LONG PANEL
# =========================================================
panel_filtered = panel[panel["cusip_id"].isin(usable_cusips)].copy()

# =========================================================
# REBUILD FILTERED COVERAGE SUMMARY FROM FILTERED PANEL
# This keeps everything internally consistent
# =========================================================
coverage_filtered = (
    panel_filtered.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        sector=("sector", "first"),
        grade=("grade", "first"),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
        n_trade_days=("trade_date", "nunique"),
        total_trades=("n_trades", "sum"),
        avg_trades_per_day=("n_trades", "mean"),
        median_price_mean=("median_price", "mean"),
        median_yield_mean=("median_yield", "mean"),
        median_price_std=("median_price", "std"),
        median_yield_std=("median_yield", "std"),
        total_quantity=("total_quantity", "sum"),
        years_to_maturity_from_first_trace=("years_to_maturity_from_first_trace", "first"),
        years_to_maturity_asof_2024_01_01=("years_to_maturity_asof_2024_01_01", "first"),
        maturity_date=("mtrty_dt", "first"),
    )
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)

# =========================================================
# BUILD FILTERED WIDE DAILY YIELD PANEL
# =========================================================
yield_panel_filtered_wide = panel_filtered.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="median_yield",
    aggfunc="first"
).sort_index()

# =========================================================
# PAIR OVERLAP TABLE
# For every retained bond pair, count overlap in trade dates
# =========================================================
presence = panel_filtered[["trade_date", "cusip_id"]].drop_duplicates().copy()
presence["present"] = 1

presence_wide = presence.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="present",
    aggfunc="max"
).fillna(0).astype(int)

cusips = list(presence_wide.columns)
overlap_rows = []

for i, c1 in enumerate(cusips):
    s1 = presence_wide[c1].values
    for c2 in cusips[i+1:]:
        s2 = presence_wide[c2].values
        overlap_days = int((s1 & s2).sum())
        overlap_rows.append({
            "cusip_1": c1,
            "cusip_2": c2,
            "overlap_days": overlap_days
        })

pair_overlap_filtered = pd.DataFrame(overlap_rows).sort_values(
    ["overlap_days", "cusip_1", "cusip_2"],
    ascending=[False, True, True]
).reset_index(drop=True)

# =========================================================
# FILTER REPORT
# =========================================================
filter_report = pd.DataFrame({
    "metric": [
        "min_trade_days",
        "min_avg_trades_per_day",
        "min_yield_std",
        "min_price_std",
        "original_unique_cusips",
        "retained_unique_cusips",
        "dropped_unique_cusips",
        "original_unique_companies",
        "retained_unique_companies",
        "original_cusip_day_rows",
        "retained_cusip_day_rows"
    ],
    "value": [
        min_trade_days,
        min_avg_trades_per_day,
        min_yield_std,
        min_price_std,
        coverage["cusip_id"].nunique(),
        coverage_filtered["cusip_id"].nunique(),
        dropped_bonds["cusip_id"].nunique(),
        coverage["company_symbol"].nunique() if "company_symbol" in coverage.columns else np.nan,
        coverage_filtered["company_symbol"].nunique() if "company_symbol" in coverage_filtered.columns else np.nan,
        len(panel),
        len(panel_filtered)
    ]
})

# =========================================================
# REASONS FOR DROP
# =========================================================
dropped_bonds = dropped_bonds.copy()
dropped_bonds["fail_trade_days"] = dropped_bonds["n_trade_days"] < min_trade_days
dropped_bonds["fail_avg_trades_per_day"] = dropped_bonds["avg_trades_per_day"] < min_avg_trades_per_day
dropped_bonds["fail_yield_std"] = dropped_bonds["median_yield_std"] < min_yield_std
dropped_bonds["fail_price_std"] = dropped_bonds["median_price_std"] < min_price_std

# =========================================================
# SAVE OUTPUTS
# =========================================================
usable_summary_file = os.path.join(output_folder, "usable_bond_universe_summary.csv")
panel_filtered_file = os.path.join(output_folder, "cusip_day_panel_filtered.csv")
yield_panel_filtered_file = os.path.join(output_folder, "cusip_day_yield_panel_filtered_wide.csv")
coverage_filtered_file = os.path.join(output_folder, "cusip_day_coverage_summary_filtered.csv")
pair_overlap_file = os.path.join(output_folder, "cusip_day_pair_overlap_filtered.csv")
dropped_bonds_file = os.path.join(output_folder, "bonds_dropped_in_usable_filter.csv")
filter_report_file = os.path.join(output_folder, "usable_bonds_filter_report.csv")

coverage_filtered.to_csv(usable_summary_file, index=False)
panel_filtered.to_csv(panel_filtered_file, index=False)
yield_panel_filtered_wide.to_csv(yield_panel_filtered_file)
coverage_filtered.to_csv(coverage_filtered_file, index=False)
pair_overlap_filtered.to_csv(pair_overlap_file, index=False)
dropped_bonds.to_csv(dropped_bonds_file, index=False)
filter_report.to_csv(filter_report_file, index=False)

# =========================================================
# CREATE ZIP FILE
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== USABLE BONDS FILTER COMPLETE ===")
print(f"Thresholds:")
print(f"  min_trade_days = {min_trade_days}")
print(f"  min_avg_trades_per_day = {min_avg_trades_per_day}")
print(f"  min_yield_std = {min_yield_std}")
print(f"  min_price_std = {min_price_std}")
print()
print(f"Original unique CUSIPs: {coverage['cusip_id'].nunique():,}")
print(f"Retained unique CUSIPs: {coverage_filtered['cusip_id'].nunique():,}")
print(f"Dropped unique CUSIPs: {dropped_bonds['cusip_id'].nunique():,}")
print()
print(f"Original unique companies: {coverage['company_symbol'].nunique() if 'company_symbol' in coverage.columns else 'N/A'}")
print(f"Retained unique companies: {coverage_filtered['company_symbol'].nunique() if 'company_symbol' in coverage_filtered.columns else 'N/A'}")
print()
print(f"Original CUSIP-day rows: {len(panel):,}")
print(f"Retained CUSIP-day rows: {len(panel_filtered):,}")
print()
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

# Optional Colab auto-download
try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== USABLE BONDS FILTER COMPLETE ===
Thresholds:
  min_trade_days = 100
  min_avg_trades_per_day = 3
  min_yield_std = 0.02
  min_price_std = 0.02

Original unique CUSIPs: 4,495
Retained unique CUSIPs: 2,331
Dropped unique CUSIPs: 2,164

Original unique companies: 233
Retained unique companies: 229

Original CUSIP-day rows: 991,938
Retained CUSIP-day rows: 919,129

Output folder created: usable_bonds_outputs_2024_2025
Zip file created: usable_bonds_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from itertools import combinations

# =========================================================
# FILE PATHS
# =========================================================
panel_file = "cusip_day_panel_filtered.csv"
treasury_2024_file = "2024Treasury.csv"
treasury_2025_file = "2025Treasury.csv"

output_folder = "spread_construction_outputs_2024_2025"
zip_filename = "spread_construction_outputs_2024_2025.zip"

# =========================================================
# CREATE OUTPUT FOLDER
# =========================================================
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def load_treasury_file(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if "Date" not in df.columns:
        raise ValueError(f"'Date' column not found in {path}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).copy()

    for col in df.columns:
        if col != "Date":
            df[col] = clean_numeric(df[col])

    return df.sort_values("Date").drop_duplicates(subset=["Date"]).reset_index(drop=True)

def tenor_to_years(col_name):
    """
    Convert Treasury column names like:
    '1 Mo', '2 Mo', '1 Yr', '2 Yr', '30 Yr'
    into numeric year values.
    """
    c = str(col_name).strip()
    if c.endswith("Mo"):
        n = float(c.replace("Mo", "").strip())
        return n / 12.0
    if c.endswith("Yr"):
        n = float(c.replace("Yr", "").strip())
        return n
    return None

def interpolate_treasury_yield(row, tenor_cols, tenor_years, maturity_years):
    """
    Linear interpolation of Treasury yield for a given remaining maturity.
    """
    if pd.isna(maturity_years):
        return np.nan

    values = row[tenor_cols].values.astype(float)
    valid = np.isfinite(values)

    if valid.sum() < 2:
        return np.nan

    x = tenor_years[valid]
    y = values[valid]

    # clamp to available range
    if maturity_years <= x.min():
        return y[x.argmin()]
    if maturity_years >= x.max():
        return y[x.argmax()]

    return np.interp(maturity_years, x, y)

# =========================================================
# LOAD FILTERED PANEL
# =========================================================
panel = pd.read_csv(panel_file, low_memory=False)

# Standardize types
panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel["first_trace_date"] = pd.to_datetime(panel["first_trace_date"], errors="coerce")
panel["mtrty_dt"] = pd.to_datetime(panel["mtrty_dt"], errors="coerce")

panel["cusip_id"] = panel["cusip_id"].astype(str).str.strip().str.upper()
panel["company_symbol"] = panel["company_symbol"].astype(str).str.strip().str.upper()
panel["issuer_nm"] = panel["issuer_nm"].astype(str).str.strip()
panel["entity_name"] = panel["entity_name"].astype(str).str.strip()
panel["sector"] = panel["sector"].astype(str).str.strip()
panel["grade"] = panel["grade"].astype(str).str.strip().str.upper()

numeric_cols = [
    "median_price", "mean_price", "median_yield", "mean_yield",
    "n_trades", "total_quantity", "vw_price", "vw_yield",
    "years_to_maturity_from_first_trace", "years_to_maturity_asof_2024_01_01"
]
for col in numeric_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

# Drop rows missing essential spread-construction fields
panel = panel.dropna(subset=["cusip_id", "trade_date", "mtrty_dt", "median_yield"]).copy()

# =========================================================
# LOAD + COMBINE TREASURY FILES
# =========================================================
t24 = load_treasury_file(treasury_2024_file)
t25 = load_treasury_file(treasury_2025_file)

treasury = (
    pd.concat([t24, t25], ignore_index=True)
    .sort_values("Date")
    .drop_duplicates(subset=["Date"])
    .reset_index(drop=True)
)

# Treasury tenor columns
tenor_cols = [c for c in treasury.columns if c != "Date"]
tenor_map = {c: tenor_to_years(c) for c in tenor_cols}
tenor_cols = [c for c in tenor_cols if tenor_map[c] is not None]
tenor_years = np.array([tenor_map[c] for c in tenor_cols], dtype=float)

# Sort tenor columns by maturity
order = np.argsort(tenor_years)
tenor_cols = [tenor_cols[i] for i in order]
tenor_years = tenor_years[order]

# Save combined treasury file
treasury.to_csv(os.path.join(output_folder, "combined_treasury_curve_2024_2025.csv"), index=False)

# =========================================================
# MERGE TREASURY ON TRADE DATE
# =========================================================
spread_panel = panel.merge(
    treasury,
    left_on="trade_date",
    right_on="Date",
    how="left"
)

spread_panel = spread_panel.drop(columns=["Date"], errors="ignore")

# =========================================================
# COMPUTE REMAINING MATURITY ON TRADE DATE
# =========================================================
spread_panel["years_to_maturity_on_trade_date"] = (
    (spread_panel["mtrty_dt"] - spread_panel["trade_date"]).dt.days / 365.25
)

# Keep only bond-days with positive remaining maturity
spread_panel = spread_panel[spread_panel["years_to_maturity_on_trade_date"] > 0].copy()

# =========================================================
# INTERPOLATE DAILY TREASURY YIELD
# =========================================================
spread_panel["interp_treasury_yield"] = spread_panel.apply(
    lambda row: interpolate_treasury_yield(
        row=row,
        tenor_cols=tenor_cols,
        tenor_years=tenor_years,
        maturity_years=row["years_to_maturity_on_trade_date"]
    ),
    axis=1
)

# =========================================================
# COMPUTE DAILY SPREADS
# Use median_yield as the main spread input
# =========================================================
spread_panel["daily_spread"] = spread_panel["median_yield"] - spread_panel["interp_treasury_yield"]

# Optional alternative spread versions
if "mean_yield" in spread_panel.columns:
    spread_panel["daily_spread_mean_yield"] = spread_panel["mean_yield"] - spread_panel["interp_treasury_yield"]
if "vw_yield" in spread_panel.columns:
    spread_panel["daily_spread_vw_yield"] = spread_panel["vw_yield"] - spread_panel["interp_treasury_yield"]

# Drop rows where spread could not be computed
spread_panel = spread_panel.dropna(subset=["interp_treasury_yield", "daily_spread"]).copy()

# =========================================================
# SAVE LONG SPREAD PANEL
# =========================================================
spread_panel_out = spread_panel[
    [
        "cusip_id",
        "company_symbol",
        "issuer_nm",
        "entity_name",
        "sector",
        "grade",
        "trade_date",
        "mtrty_dt",
        "first_trace_date",
        "years_to_maturity_from_first_trace",
        "years_to_maturity_asof_2024_01_01",
        "years_to_maturity_on_trade_date",
        "median_price",
        "mean_price",
        "median_yield",
        "mean_yield",
        "vw_yield",
        "n_trades",
        "total_quantity",
        "interp_treasury_yield",
        "daily_spread",
        "daily_spread_mean_yield",
        "daily_spread_vw_yield",
    ]
].copy()

spread_panel_file = os.path.join(output_folder, "cusip_day_spread_panel.csv")
spread_panel_out.to_csv(spread_panel_file, index=False)

# =========================================================
# WIDE SPREAD PANEL
# =========================================================
spread_panel_wide = spread_panel_out.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="daily_spread",
    aggfunc="first"
).sort_index()

spread_panel_wide_file = os.path.join(output_folder, "cusip_day_spread_panel_wide.csv")
spread_panel_wide.to_csv(spread_panel_wide_file)

# =========================================================
# SPREAD SUMMARY
# =========================================================
spread_summary = (
    spread_panel_out.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        sector=("sector", "first"),
        grade=("grade", "first"),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
        n_trade_days=("trade_date", "nunique"),
        avg_trades_per_day=("n_trades", "mean"),
        total_trades=("n_trades", "sum"),
        mean_spread=("daily_spread", "mean"),
        median_spread=("daily_spread", "median"),
        std_spread=("daily_spread", "std"),
        min_spread=("daily_spread", "min"),
        max_spread=("daily_spread", "max"),
        pct_negative_spread=("daily_spread", lambda x: np.mean(x < 0) if len(x) > 0 else np.nan),
        avg_remaining_maturity=("years_to_maturity_on_trade_date", "mean"),
        min_remaining_maturity=("years_to_maturity_on_trade_date", "min"),
        max_remaining_maturity=("years_to_maturity_on_trade_date", "max"),
        maturity_date=("mtrty_dt", "first"),
    )
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)

spread_summary_file = os.path.join(output_folder, "cusip_day_spread_summary.csv")
spread_summary.to_csv(spread_summary_file, index=False)

# =========================================================
# PAIR OVERLAP TABLE FOR SPREAD PANEL
# =========================================================
presence = spread_panel_out[["trade_date", "cusip_id"]].drop_duplicates().copy()
presence["present"] = 1

presence_wide = presence.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="present",
    aggfunc="max"
).fillna(0).astype(int)

cusips = list(presence_wide.columns)
overlap_rows = []

for i, c1 in enumerate(cusips):
    s1 = presence_wide[c1].values
    for c2 in cusips[i+1:]:
        s2 = presence_wide[c2].values
        overlap_days = int((s1 & s2).sum())
        overlap_rows.append({
            "cusip_1": c1,
            "cusip_2": c2,
            "overlap_days": overlap_days
        })

spread_pair_overlap = pd.DataFrame(overlap_rows).sort_values(
    ["overlap_days", "cusip_1", "cusip_2"],
    ascending=[False, True, True]
).reset_index(drop=True)

spread_pair_overlap_file = os.path.join(output_folder, "cusip_day_spread_pair_overlap.csv")
spread_pair_overlap.to_csv(spread_pair_overlap_file, index=False)

# =========================================================
# CONSTRUCTION REPORT
# =========================================================
report = pd.DataFrame({
    "metric": [
        "input_cusip_day_rows",
        "output_spread_rows",
        "unique_cusips",
        "unique_companies",
        "start_date",
        "end_date",
        "min_remaining_maturity_on_trade_date",
        "max_remaining_maturity_on_trade_date",
        "mean_daily_spread",
        "median_daily_spread"
    ],
    "value": [
        len(panel),
        len(spread_panel_out),
        spread_panel_out["cusip_id"].nunique(),
        spread_panel_out["company_symbol"].nunique(),
        spread_panel_out["trade_date"].min(),
        spread_panel_out["trade_date"].max(),
        spread_panel_out["years_to_maturity_on_trade_date"].min(),
        spread_panel_out["years_to_maturity_on_trade_date"].max(),
        spread_panel_out["daily_spread"].mean(),
        spread_panel_out["daily_spread"].median()
    ]
})

report_file = os.path.join(output_folder, "spread_construction_report.csv")
report.to_csv(report_file, index=False)

# =========================================================
# CREATE ZIP FILE
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== SPREAD CONSTRUCTION COMPLETE ===")
print(f"Input CUSIP-day rows: {len(panel):,}")
print(f"Output spread rows: {len(spread_panel_out):,}")
print(f"Unique CUSIPs: {spread_panel_out['cusip_id'].nunique():,}")
print(f"Unique companies: {spread_panel_out['company_symbol'].nunique():,}")
print(f"Date range: {spread_panel_out['trade_date'].min()} to {spread_panel_out['trade_date'].max()}")
print(f"Remaining maturity on trade date range: {spread_panel_out['years_to_maturity_on_trade_date'].min():.4f} to {spread_panel_out['years_to_maturity_on_trade_date'].max():.4f}")
print(f"Mean daily spread: {spread_panel_out['daily_spread'].mean():.6f}")
print(f"Median daily spread: {spread_panel_out['daily_spread'].median():.6f}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

# Optional Colab auto-download
try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== SPREAD CONSTRUCTION COMPLETE ===
Input CUSIP-day rows: 919,129
Output spread rows: 916,832
Unique CUSIPs: 2,331
Unique companies: 229
Date range: 2024-01-02 00:00:00 to 2025-12-31 00:00:00
Remaining maturity on trade date range: 1.0021 to 9.9959
Mean daily spread: 0.674080
Median daily spread: 0.645146
Output folder created: spread_construction_outputs_2024_2025
Zip file created: spread_construction_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# FILE PATHS
# =========================================================
spread_panel_file = "cusip_day_spread_panel.csv"
spread_summary_file = "cusip_day_spread_summary.csv"

output_folder = "spread_cleaning_outputs_2024_2025"
zip_filename = "spread_cleaning_outputs_2024_2025.zip"

# =========================================================
# CREATE OUTPUT FOLDER
# =========================================================
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# PARAMETERS
# These are deliberately conservative so you do not over-prune
# the sample before validation / pair construction.
# =========================================================
min_trade_days = 100
max_pct_negative_spread = 0.25
max_abs_mean_spread = 5.0
max_std_spread = 1.50

# Within-bond winsorization
winsor_lower = 0.01
winsor_upper = 0.99

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    """
    Winsorize a numeric series using within-series quantiles.
    """
    s = pd.to_numeric(s, errors="coerce")
    valid = s.dropna()
    if len(valid) == 0:
        return s
    lo = valid.quantile(lower_q)
    hi = valid.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

def clean_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# =========================================================
# LOAD INPUT FILES
# =========================================================
spread_panel = pd.read_csv(spread_panel_file, low_memory=False)
spread_summary = pd.read_csv(spread_summary_file, low_memory=False)

# =========================================================
# STANDARDIZE TYPES
# =========================================================
date_cols_panel = ["trade_date", "mtrty_dt", "first_trace_date"]
for col in date_cols_panel:
    if col in spread_panel.columns:
        spread_panel[col] = pd.to_datetime(spread_panel[col], errors="coerce")

date_cols_summary = ["first_trade_date", "last_trade_date", "maturity_date"]
for col in date_cols_summary:
    if col in spread_summary.columns:
        spread_summary[col] = pd.to_datetime(spread_summary[col], errors="coerce")

str_cols_panel = [
    "cusip_id", "company_symbol", "issuer_nm", "entity_name", "sector", "grade"
]
for col in str_cols_panel:
    if col in spread_panel.columns:
        spread_panel[col] = spread_panel[col].astype(str).str.strip()

str_cols_summary = [
    "cusip_id", "company_symbol", "issuer_name", "entity_name", "sector", "grade"
]
for col in str_cols_summary:
    if col in spread_summary.columns:
        spread_summary[col] = spread_summary[col].astype(str).str.strip()

numeric_cols_panel = [
    "years_to_maturity_from_first_trace",
    "years_to_maturity_asof_2024_01_01",
    "years_to_maturity_on_trade_date",
    "median_price",
    "mean_price",
    "median_yield",
    "mean_yield",
    "vw_yield",
    "n_trades",
    "total_quantity",
    "interp_treasury_yield",
    "daily_spread",
    "daily_spread_mean_yield",
    "daily_spread_vw_yield",
]
spread_panel = clean_numeric(spread_panel, numeric_cols_panel)

numeric_cols_summary = [
    "n_trade_days",
    "avg_trades_per_day",
    "total_trades",
    "mean_spread",
    "median_spread",
    "std_spread",
    "min_spread",
    "max_spread",
    "pct_negative_spread",
    "avg_remaining_maturity",
    "min_remaining_maturity",
    "max_remaining_maturity",
]
spread_summary = clean_numeric(spread_summary, numeric_cols_summary)

# Drop rows missing essential fields
spread_panel = spread_panel.dropna(subset=["cusip_id", "trade_date", "daily_spread"]).copy()
spread_summary = spread_summary.dropna(subset=["cusip_id"]).copy()

# =========================================================
# FLAG BONDS FOR REVIEW / DROP
# =========================================================
flag_rows = []

for _, row in spread_summary.iterrows():
    reasons = []

    if pd.notna(row.get("n_trade_days")) and row["n_trade_days"] < min_trade_days:
        reasons.append(f"n_trade_days_below_{min_trade_days}")

    if pd.notna(row.get("pct_negative_spread")) and row["pct_negative_spread"] > max_pct_negative_spread:
        reasons.append(f"pct_negative_spread_above_{max_pct_negative_spread}")

    if pd.notna(row.get("mean_spread")) and abs(row["mean_spread"]) > max_abs_mean_spread:
        reasons.append(f"abs_mean_spread_above_{max_abs_mean_spread}")

    if pd.notna(row.get("std_spread")) and row["std_spread"] > max_std_spread:
        reasons.append(f"std_spread_above_{max_std_spread}")

    flag_rows.append({
        "cusip_id": row["cusip_id"],
        "company_symbol": row.get("company_symbol"),
        "issuer_name": row.get("issuer_name"),
        "entity_name": row.get("entity_name"),
        "sector": row.get("sector"),
        "grade": row.get("grade"),
        "n_trade_days": row.get("n_trade_days"),
        "avg_trades_per_day": row.get("avg_trades_per_day"),
        "total_trades": row.get("total_trades"),
        "mean_spread": row.get("mean_spread"),
        "median_spread": row.get("median_spread"),
        "std_spread": row.get("std_spread"),
        "min_spread": row.get("min_spread"),
        "max_spread": row.get("max_spread"),
        "pct_negative_spread": row.get("pct_negative_spread"),
        "n_flag_reasons": len(reasons),
        "drop_from_model": int(len(reasons) > 0),
        "flag_reasons": "; ".join(reasons) if reasons else ""
    })

flag_df = pd.DataFrame(flag_rows)

spread_bonds_flagged_for_review = (
    flag_df[flag_df["drop_from_model"] == 1]
    .sort_values(["n_flag_reasons", "pct_negative_spread", "std_spread", "mean_spread"], ascending=[False, False, False, False])
    .reset_index(drop=True)
)

spread_bonds_kept_for_model = (
    flag_df[flag_df["drop_from_model"] == 0]
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)

flagged_file = os.path.join(output_folder, "spread_bonds_flagged_for_review.csv")
kept_file = os.path.join(output_folder, "spread_bonds_kept_for_model.csv")

spread_bonds_flagged_for_review.to_csv(flagged_file, index=False)
spread_bonds_kept_for_model.to_csv(kept_file, index=False)

# =========================================================
# FILTER PANEL TO KEPT BONDS
# =========================================================
kept_cusips = set(spread_bonds_kept_for_model["cusip_id"])
spread_panel_filtered = spread_panel[spread_panel["cusip_id"].isin(kept_cusips)].copy()

# =========================================================
# PRESERVE PRE-WINSORIZATION SPREADS FOR IMPACT CHECKS
# =========================================================
spread_panel_filtered["daily_spread_raw"] = spread_panel_filtered["daily_spread"]
if "daily_spread_mean_yield" in spread_panel_filtered.columns:
    spread_panel_filtered["daily_spread_mean_yield_raw"] = spread_panel_filtered["daily_spread_mean_yield"]
if "daily_spread_vw_yield" in spread_panel_filtered.columns:
    spread_panel_filtered["daily_spread_vw_yield_raw"] = spread_panel_filtered["daily_spread_vw_yield"]

# =========================================================
# WITHIN-BOND WINSORIZATION
# =========================================================
spread_panel_filtered = spread_panel_filtered.sort_values(["cusip_id", "trade_date"]).copy()

spread_panel_filtered["daily_spread"] = (
    spread_panel_filtered.groupby("cusip_id")["daily_spread"]
    .transform(lambda s: winsorize_series(s, winsor_lower, winsor_upper))
)

if "daily_spread_mean_yield" in spread_panel_filtered.columns:
    spread_panel_filtered["daily_spread_mean_yield"] = (
        spread_panel_filtered.groupby("cusip_id")["daily_spread_mean_yield"]
        .transform(lambda s: winsorize_series(s, winsor_lower, winsor_upper))
    )

if "daily_spread_vw_yield" in spread_panel_filtered.columns:
    spread_panel_filtered["daily_spread_vw_yield"] = (
        spread_panel_filtered.groupby("cusip_id")["daily_spread_vw_yield"]
        .transform(lambda s: winsorize_series(s, winsor_lower, winsor_upper))
    )

# =========================================================
# WINSORIZATION IMPACT REPORT
# =========================================================
impact_rows = []

for cusip, g in spread_panel_filtered.groupby("cusip_id"):
    raw = pd.to_numeric(g["daily_spread_raw"], errors="coerce")
    clean = pd.to_numeric(g["daily_spread"], errors="coerce")

    changed_mask = (~raw.isna()) & (~clean.isna()) & (np.abs(raw - clean) > 1e-12)

    impact_rows.append({
        "cusip_id": cusip,
        "company_symbol": g["company_symbol"].iloc[0] if "company_symbol" in g.columns else np.nan,
        "issuer_name": g["issuer_nm"].iloc[0] if "issuer_nm" in g.columns else np.nan,
        "sector": g["sector"].iloc[0] if "sector" in g.columns else np.nan,
        "n_rows": len(g),
        "n_rows_changed": int(changed_mask.sum()),
        "pct_rows_changed": float(changed_mask.mean()) if len(g) > 0 else np.nan,
        "raw_mean_spread": raw.mean(),
        "winsorized_mean_spread": clean.mean(),
        "raw_std_spread": raw.std(),
        "winsorized_std_spread": clean.std(),
        "max_abs_change": float(np.max(np.abs(raw[changed_mask] - clean[changed_mask]))) if changed_mask.any() else 0.0
    })

spread_winsorization_impact = pd.DataFrame(impact_rows).sort_values(
    ["pct_rows_changed", "max_abs_change", "n_rows_changed"],
    ascending=[False, False, False]
).reset_index(drop=True)

winsor_impact_file = os.path.join(output_folder, "spread_winsorization_impact.csv")
spread_winsorization_impact.to_csv(winsor_impact_file, index=False)

# =========================================================
# CLEANED LONG SPREAD PANEL
# =========================================================
cleaned_panel_file = os.path.join(output_folder, "cusip_day_spread_panel_cleaned.csv")
spread_panel_filtered.to_csv(cleaned_panel_file, index=False)

# =========================================================
# CLEANED WIDE SPREAD PANEL
# =========================================================
spread_panel_cleaned_wide = spread_panel_filtered.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="daily_spread",
    aggfunc="first"
).sort_index()

cleaned_wide_file = os.path.join(output_folder, "cusip_day_spread_panel_cleaned_wide.csv")
spread_panel_cleaned_wide.to_csv(cleaned_wide_file)

# =========================================================
# CLEANED SPREAD SUMMARY
# =========================================================
spread_summary_cleaned = (
    spread_panel_filtered.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        sector=("sector", "first"),
        grade=("grade", "first"),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
        n_trade_days=("trade_date", "nunique"),
        avg_trades_per_day=("n_trades", "mean"),
        total_trades=("n_trades", "sum"),
        mean_spread=("daily_spread", "mean"),
        median_spread=("daily_spread", "median"),
        std_spread=("daily_spread", "std"),
        min_spread=("daily_spread", "min"),
        max_spread=("daily_spread", "max"),
        pct_negative_spread=("daily_spread", lambda x: np.mean(pd.to_numeric(x, errors="coerce") < 0) if len(x) > 0 else np.nan),
        avg_remaining_maturity=("years_to_maturity_on_trade_date", "mean"),
        min_remaining_maturity=("years_to_maturity_on_trade_date", "min"),
        max_remaining_maturity=("years_to_maturity_on_trade_date", "max"),
        maturity_date=("mtrty_dt", "first"),
    )
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)

cleaned_summary_file = os.path.join(output_folder, "cusip_day_spread_summary_cleaned.csv")
spread_summary_cleaned.to_csv(cleaned_summary_file, index=False)

# =========================================================
# CLEANED PAIR OVERLAP TABLE
# =========================================================
presence = spread_panel_filtered[["trade_date", "cusip_id"]].drop_duplicates().copy()
presence["present"] = 1

presence_wide = presence.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="present",
    aggfunc="max"
).fillna(0).astype(int)

cusips = list(presence_wide.columns)
overlap_rows = []

for i, c1 in enumerate(cusips):
    s1 = presence_wide[c1].values
    for c2 in cusips[i + 1:]:
        s2 = presence_wide[c2].values
        overlap_days = int((s1 & s2).sum())
        overlap_rows.append({
            "cusip_1": c1,
            "cusip_2": c2,
            "overlap_days": overlap_days
        })

spread_pair_overlap_cleaned = (
    pd.DataFrame(overlap_rows)
    .sort_values(["overlap_days", "cusip_1", "cusip_2"], ascending=[False, True, True])
    .reset_index(drop=True)
)

cleaned_overlap_file = os.path.join(output_folder, "cusip_day_spread_pair_overlap_cleaned.csv")
spread_pair_overlap_cleaned.to_csv(cleaned_overlap_file, index=False)

# =========================================================
# CLEANING REPORT
# =========================================================
cleaning_report = pd.DataFrame({
    "metric": [
        "input_spread_rows",
        "input_unique_cusips",
        "input_unique_companies",
        "flagged_cusips",
        "kept_cusips",
        "output_cleaned_rows",
        "output_cleaned_unique_cusips",
        "output_cleaned_unique_companies",
        "input_mean_spread",
        "output_mean_spread",
        "input_median_spread",
        "output_median_spread",
        "winsor_lower_quantile",
        "winsor_upper_quantile",
        "min_trade_days_threshold",
        "max_pct_negative_spread_threshold",
        "max_abs_mean_spread_threshold",
        "max_std_spread_threshold"
    ],
    "value": [
        len(spread_panel),
        spread_panel["cusip_id"].nunique(),
        spread_panel["company_symbol"].nunique(),
        spread_bonds_flagged_for_review["cusip_id"].nunique(),
        spread_bonds_kept_for_model["cusip_id"].nunique(),
        len(spread_panel_filtered),
        spread_panel_filtered["cusip_id"].nunique(),
        spread_panel_filtered["company_symbol"].nunique(),
        spread_panel["daily_spread"].mean(),
        spread_panel_filtered["daily_spread"].mean(),
        spread_panel["daily_spread"].median(),
        spread_panel_filtered["daily_spread"].median(),
        winsor_lower,
        winsor_upper,
        min_trade_days,
        max_pct_negative_spread,
        max_abs_mean_spread,
        max_std_spread
    ]
})

cleaning_report_file = os.path.join(output_folder, "spread_cleaning_report.csv")
cleaning_report.to_csv(cleaning_report_file, index=False)

# =========================================================
# CREATE ZIP FILE
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== SPREAD CLEANING COMPLETE ===")
print(f"Input spread rows: {len(spread_panel):,}")
print(f"Input unique CUSIPs: {spread_panel['cusip_id'].nunique():,}")
print(f"Flagged CUSIPs: {spread_bonds_flagged_for_review['cusip_id'].nunique():,}")
print(f"Kept CUSIPs: {spread_bonds_kept_for_model['cusip_id'].nunique():,}")
print(f"Output cleaned rows: {len(spread_panel_filtered):,}")
print(f"Output cleaned unique companies: {spread_panel_filtered['company_symbol'].nunique():,}")
print(f"Input mean spread: {spread_panel['daily_spread'].mean():.6f}")
print(f"Output mean spread: {spread_panel_filtered['daily_spread'].mean():.6f}")
print(f"Input median spread: {spread_panel['daily_spread'].median():.6f}")
print(f"Output median spread: {spread_panel_filtered['daily_spread'].median():.6f}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

# Optional Colab auto-download
try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== SPREAD CLEANING COMPLETE ===
Input spread rows: 916,832
Input unique CUSIPs: 2,331
Flagged CUSIPs: 34
Kept CUSIPs: 2,297
Output cleaned rows: 910,032
Output cleaned unique companies: 228
Input mean spread: 0.674080
Output mean spread: 0.685812
Input median spread: 0.645146
Output median spread: 0.646289
Output folder created: spread_cleaning_outputs_2024_2025
Zip file created: spread_cleaning_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# FILE PATHS
# =========================================================
cleaned_panel_file = "cusip_day_spread_panel_cleaned.csv"
cleaned_summary_file = "cusip_day_spread_summary_cleaned.csv"
cleaned_overlap_file = "cusip_day_spread_pair_overlap_cleaned.csv"
flagged_file = "spread_bonds_flagged_for_review.csv"
winsor_file = "spread_winsorization_impact.csv"

output_folder = "spread_validation_outputs_2024_2025"
zip_filename = "spread_validation_outputs_2024_2025.zip"

os.makedirs(output_folder, exist_ok=True)

# =========================================================
# PARAMETERS
# =========================================================
sample_n_math_check = 2000
low_overlap_threshold = 120
negative_spread_review_threshold = 0.25
winsorization_review_threshold = 0.10
outlier_z_threshold = 6.0

# =========================================================
# LOAD FILES
# =========================================================
panel = pd.read_csv(cleaned_panel_file, low_memory=False)
summary = pd.read_csv(cleaned_summary_file, low_memory=False)
overlap = pd.read_csv(cleaned_overlap_file, low_memory=False)
flagged = pd.read_csv(flagged_file, low_memory=False)
winsor = pd.read_csv(winsor_file, low_memory=False)

# =========================================================
# STANDARDIZE TYPES
# =========================================================
date_cols_panel = ["trade_date", "mtrty_dt", "first_trace_date"]
for col in date_cols_panel:
    if col in panel.columns:
        panel[col] = pd.to_datetime(panel[col], errors="coerce")

date_cols_summary = ["first_trade_date", "last_trade_date", "maturity_date"]
for col in date_cols_summary:
    if col in summary.columns:
        summary[col] = pd.to_datetime(summary[col], errors="coerce")

str_cols_panel = ["cusip_id", "company_symbol", "issuer_nm", "entity_name", "sector", "grade"]
for col in str_cols_panel:
    if col in panel.columns:
        panel[col] = panel[col].astype(str).str.strip()

str_cols_summary = ["cusip_id", "company_symbol", "issuer_name", "entity_name", "sector", "grade"]
for col in str_cols_summary:
    if col in summary.columns:
        summary[col] = summary[col].astype(str).str.strip()

numeric_cols_panel = [
    "years_to_maturity_from_first_trace",
    "years_to_maturity_asof_2024_01_01",
    "years_to_maturity_on_trade_date",
    "median_price",
    "mean_price",
    "median_yield",
    "mean_yield",
    "vw_yield",
    "n_trades",
    "total_quantity",
    "interp_treasury_yield",
    "daily_spread",
    "daily_spread_raw",
    "daily_spread_mean_yield",
    "daily_spread_mean_yield_raw",
    "daily_spread_vw_yield",
    "daily_spread_vw_yield_raw",
]
for col in numeric_cols_panel:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

numeric_cols_summary = [
    "n_trade_days",
    "avg_trades_per_day",
    "total_trades",
    "mean_spread",
    "median_spread",
    "std_spread",
    "min_spread",
    "max_spread",
    "pct_negative_spread",
    "avg_remaining_maturity",
    "min_remaining_maturity",
    "max_remaining_maturity",
]
for col in numeric_cols_summary:
    if col in summary.columns:
        summary[col] = pd.to_numeric(summary[col], errors="coerce")

# =========================================================
# A. SPREAD MATH VALIDATION
# Use daily_spread_raw if it exists, because cleaned daily_spread
# may have been winsorized and will no longer exactly equal
# median_yield - interp_treasury_yield
# =========================================================
spread_col_for_math_check = "daily_spread_raw" if "daily_spread_raw" in panel.columns else "daily_spread"

math_check = panel.dropna(subset=["median_yield", "interp_treasury_yield", spread_col_for_math_check]).copy()
if len(math_check) > sample_n_math_check:
    math_check = math_check.sample(sample_n_math_check, random_state=42)

math_check["spread_used_for_validation"] = math_check[spread_col_for_math_check]
math_check["recomputed_spread"] = math_check["median_yield"] - math_check["interp_treasury_yield"]
math_check["spread_math_error"] = math_check["spread_used_for_validation"] - math_check["recomputed_spread"]
math_check["abs_spread_math_error"] = math_check["spread_math_error"].abs()

spread_math_summary = pd.DataFrame({
    "metric": [
        "spread_column_used",
        "rows_checked",
        "mean_abs_error",
        "median_abs_error",
        "max_abs_error",
        "pct_rows_abs_error_gt_1e_10",
        "pct_rows_abs_error_gt_1e_08",
        "pct_rows_abs_error_gt_1e_06",
    ],
    "value": [
        spread_col_for_math_check,
        len(math_check),
        math_check["abs_spread_math_error"].mean(),
        math_check["abs_spread_math_error"].median(),
        math_check["abs_spread_math_error"].max(),
        np.mean(math_check["abs_spread_math_error"] > 1e-10),
        np.mean(math_check["abs_spread_math_error"] > 1e-8),
        np.mean(math_check["abs_spread_math_error"] > 1e-6),
    ]
})
spread_math_summary.to_csv(os.path.join(output_folder, "validation_spread_math_summary.csv"), index=False)

# =========================================================
# B. MATURITY MATH VALIDATION
# years_to_maturity_on_trade_date == (mtrty_dt - trade_date)/365.25
# =========================================================
maturity_check = panel.dropna(subset=["trade_date", "mtrty_dt", "years_to_maturity_on_trade_date"]).copy()
if len(maturity_check) > sample_n_math_check:
    maturity_check = maturity_check.sample(sample_n_math_check, random_state=42)

maturity_check["recomputed_years_to_maturity"] = (
    (maturity_check["mtrty_dt"] - maturity_check["trade_date"]).dt.days / 365.25
)
maturity_check["maturity_math_error"] = (
    maturity_check["years_to_maturity_on_trade_date"] - maturity_check["recomputed_years_to_maturity"]
)
maturity_check["abs_maturity_math_error"] = maturity_check["maturity_math_error"].abs()

maturity_math_summary = pd.DataFrame({
    "metric": [
        "rows_checked",
        "mean_abs_error",
        "median_abs_error",
        "max_abs_error",
        "pct_rows_abs_error_gt_1e_10",
        "pct_rows_abs_error_gt_1e_08",
        "pct_rows_abs_error_gt_1e_06",
    ],
    "value": [
        len(maturity_check),
        maturity_check["abs_maturity_math_error"].mean(),
        maturity_check["abs_maturity_math_error"].median(),
        maturity_check["abs_maturity_math_error"].max(),
        np.mean(maturity_check["abs_maturity_math_error"] > 1e-10),
        np.mean(maturity_check["abs_maturity_math_error"] > 1e-8),
        np.mean(maturity_check["abs_maturity_math_error"] > 1e-6),
    ]
})
maturity_math_summary.to_csv(os.path.join(output_folder, "validation_maturity_math_summary.csv"), index=False)

# =========================================================
# C. ISSUER / SYMBOL CONSISTENCY
# =========================================================
symbol_map = (
    panel.groupby("cusip_id", as_index=False)
    .agg(
        n_company_symbols=("company_symbol", "nunique"),
        n_issuer_names=("issuer_nm", "nunique"),
        n_entity_names=("entity_name", "nunique"),
        company_symbol_example=("company_symbol", "first"),
        issuer_name_example=("issuer_nm", "first"),
        entity_name_example=("entity_name", "first"),
    )
)

symbol_mismatches = symbol_map[
    (symbol_map["n_company_symbols"] > 1) |
    (symbol_map["n_issuer_names"] > 1) |
    (symbol_map["n_entity_names"] > 1)
].copy()

symbol_mismatches.to_csv(os.path.join(output_folder, "validation_company_symbol_mismatches.csv"), index=False)

# =========================================================
# D. NEGATIVE SPREAD REVIEW
# =========================================================
negative_spread_bonds = summary[
    summary["pct_negative_spread"] > negative_spread_review_threshold
].copy().sort_values(
    ["pct_negative_spread", "mean_spread", "std_spread"],
    ascending=[False, True, False]
)

negative_spread_bonds.to_csv(os.path.join(output_folder, "validation_negative_spread_bonds.csv"), index=False)

# =========================================================
# E. OUTLIER FLAGS WITHIN BOND
# z-score within CUSIP on cleaned daily_spread
# =========================================================
def flag_within_bond_outliers(g):
    g = g.copy()
    mu = g["daily_spread"].mean()
    sd = g["daily_spread"].std()
    if pd.isna(sd) or sd == 0:
        g["spread_zscore"] = np.nan
        g["outlier_flag"] = 0
    else:
        g["spread_zscore"] = (g["daily_spread"] - mu) / sd
        g["outlier_flag"] = (g["spread_zscore"].abs() > outlier_z_threshold).astype(int)
    return g

panel_outliers = (
    panel.groupby("cusip_id", group_keys=False)
    .apply(flag_within_bond_outliers)
    .reset_index(drop=True)
)

outlier_flags = (
    panel_outliers.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        sector=("sector", "first"),
        n_rows=("trade_date", "size"),
        n_outlier_rows=("outlier_flag", "sum"),
        pct_outlier_rows=("outlier_flag", "mean"),
        max_abs_zscore=("spread_zscore", lambda x: np.nanmax(np.abs(x)) if len(x.dropna()) > 0 else np.nan),
    )
    .sort_values(["pct_outlier_rows", "max_abs_zscore"], ascending=[False, False])
    .reset_index(drop=True)
)

outlier_flags.to_csv(os.path.join(output_folder, "validation_spread_outlier_flags.csv"), index=False)

# =========================================================
# F. WINSORIZATION REVIEW
# =========================================================
winsor_flags = winsor[winsor["pct_rows_changed"] > winsorization_review_threshold].copy()
winsor_flags = winsor_flags.sort_values(
    ["pct_rows_changed", "max_abs_change", "n_rows_changed"],
    ascending=[False, False, False]
).reset_index(drop=True)

winsor_flags.to_csv(os.path.join(output_folder, "validation_winsorization_flags.csv"), index=False)

# =========================================================
# G. PAIR OVERLAP REVIEW
# =========================================================
overlap_summary = pd.DataFrame({
    "metric": [
        "n_pairs",
        "max_overlap_days",
        "mean_overlap_days",
        "median_overlap_days",
        "pct_pairs_below_low_overlap_threshold",
        "low_overlap_threshold",
    ],
    "value": [
        len(overlap),
        overlap["overlap_days"].max(),
        overlap["overlap_days"].mean(),
        overlap["overlap_days"].median(),
        np.mean(overlap["overlap_days"] < low_overlap_threshold),
        low_overlap_threshold,
    ]
})
overlap_summary.to_csv(os.path.join(output_folder, "validation_pair_overlap_summary.csv"), index=False)

low_overlap_pairs = overlap[overlap["overlap_days"] < low_overlap_threshold].copy().sort_values(
    ["overlap_days", "cusip_1", "cusip_2"], ascending=[True, True, True]
)
low_overlap_pairs.to_csv(os.path.join(output_folder, "validation_low_overlap_pairs.csv"), index=False)

# =========================================================
# H. OVERALL VALIDATION REPORT
# =========================================================
validation_report = pd.DataFrame({
    "metric": [
        "cleaned_rows",
        "cleaned_unique_cusips",
        "cleaned_unique_companies",
        "flagged_bonds_from_cleaning_stage",
        "bonds_with_symbol_mismatches",
        "bonds_with_high_negative_spread_share",
        "bonds_with_high_winsorization_share",
        "pairs_below_low_overlap_threshold",
        "spread_math_max_abs_error",
        "maturity_math_max_abs_error"
    ],
    "value": [
        len(panel),
        panel["cusip_id"].nunique(),
        panel["company_symbol"].nunique(),
        flagged["cusip_id"].nunique(),
        symbol_mismatches["cusip_id"].nunique(),
        negative_spread_bonds["cusip_id"].nunique(),
        winsor_flags["cusip_id"].nunique() if "cusip_id" in winsor_flags.columns else 0,
        len(low_overlap_pairs),
        spread_math_summary.loc[spread_math_summary["metric"] == "max_abs_error", "value"].iloc[0],
        maturity_math_summary.loc[maturity_math_summary["metric"] == "max_abs_error", "value"].iloc[0],
    ]
})
validation_report.to_csv(os.path.join(output_folder, "validation_report.csv"), index=False)

# =========================================================
# ZIP OUTPUTS
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== SPREAD VALIDATION COMPLETE ===")
print(f"Cleaned rows: {len(panel):,}")
print(f"Cleaned unique CUSIPs: {panel['cusip_id'].nunique():,}")
print(f"Cleaned unique companies: {panel['company_symbol'].nunique():,}")
print(f"Bonds with symbol mismatches: {symbol_mismatches['cusip_id'].nunique():,}")
print(f"Bonds with high negative spread share: {negative_spread_bonds['cusip_id'].nunique():,}")
print(f"Bonds with high winsorization share: {winsor_flags['cusip_id'].nunique() if 'cusip_id' in winsor_flags.columns else 0:,}")
print(f"Pairs below overlap threshold ({low_overlap_threshold}): {len(low_overlap_pairs):,}")
print(f"Spread math column used: {spread_col_for_math_check}")
print(f"Spread math max abs error: {spread_math_summary.loc[spread_math_summary['metric']=='max_abs_error','value'].iloc[0]:.12f}")
print(f"Maturity math max abs error: {maturity_math_summary.loc[maturity_math_summary['metric']=='max_abs_error','value'].iloc[0]:.12f}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

/tmp/ipykernel_19715/4002605409.py:234: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(flag_within_bond_outliers)


=== SPREAD VALIDATION COMPLETE ===
Cleaned rows: 910,032
Cleaned unique CUSIPs: 2,297
Cleaned unique companies: 228
Bonds with symbol mismatches: 0
Bonds with high negative spread share: 0
Bonds with high winsorization share: 0
Pairs below overlap threshold (120): 314,648
Spread math column used: daily_spread_raw
Spread math max abs error: 0.000000000000
Maturity math max abs error: 0.000000000000
Output folder created: spread_validation_outputs_2024_2025
Zip file created: spread_validation_outputs_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# OPTIONAL: SHAPE-PRESERVING TREASURY INTERPOLATION
# Rebuild spread panel using PCHIP instead of linear interpolation
# =========================================================

try:
    from scipy.interpolate import PchipInterpolator
except ImportError:
    raise ImportError(
        "SciPy is required for the upgraded interpolation script. "
        "Install it first in your environment, then rerun this block."
    )

# =========================================================
# FILE PATHS
# =========================================================
panel_file = "cusip_day_panel_filtered.csv"
treasury_2024_file = "2024Treasury.csv"
treasury_2025_file = "2025Treasury.csv"

output_folder = "spread_construction_outputs_pchip_2024_2025"
zip_filename = "spread_construction_outputs_pchip_2024_2025.zip"

os.makedirs(output_folder, exist_ok=True)

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def load_treasury_file(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if "Date" not in df.columns:
        raise ValueError(f"'Date' column not found in {path}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).copy()

    for col in df.columns:
        if col != "Date":
            df[col] = clean_numeric(df[col])

    return df.sort_values("Date").drop_duplicates(subset=["Date"]).reset_index(drop=True)

def tenor_to_years(col_name):
    c = str(col_name).strip()

    # support '1 Mo', '2 Mo', '1 Yr', '2 Yr'
    if c.endswith("Mo"):
        n = float(c.replace("Mo", "").strip())
        return n / 12.0
    if c.endswith("Yr"):
        n = float(c.replace("Yr", "").strip())
        return n

    # support possible alternate labels such as '1.5 Month', '6 Month', '2 Year'
    if c.endswith("Month"):
        n = float(c.replace("Month", "").strip())
        return n / 12.0
    if c.endswith("Months"):
        n = float(c.replace("Months", "").strip())
        return n / 12.0
    if c.endswith("Year"):
        n = float(c.replace("Year", "").strip())
        return n
    if c.endswith("Years"):
        n = float(c.replace("Years", "").strip())
        return n

    return None

def interpolate_treasury_yield_pchip(row, tenor_cols, tenor_years, maturity_years):
    if pd.isna(maturity_years):
        return np.nan

    values = row[tenor_cols].values.astype(float)
    valid = np.isfinite(values)

    if valid.sum() < 2:
        return np.nan

    x = tenor_years[valid]
    y = values[valid]

    # clamp outside available range
    if maturity_years <= x.min():
        return y[np.argmin(x)]
    if maturity_years >= x.max():
        return y[np.argmax(x)]

    # PCHIP shape-preserving cubic
    try:
        f = PchipInterpolator(x, y, extrapolate=False)
        val = f(maturity_years)
        return float(val) if np.isfinite(val) else np.nan
    except Exception:
        return np.nan

# =========================================================
# LOAD FILTERED PANEL
# =========================================================
panel = pd.read_csv(panel_file, low_memory=False)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel["first_trace_date"] = pd.to_datetime(panel["first_trace_date"], errors="coerce")
panel["mtrty_dt"] = pd.to_datetime(panel["mtrty_dt"], errors="coerce")

panel["cusip_id"] = panel["cusip_id"].astype(str).str.strip().str.upper()
panel["company_symbol"] = panel["company_symbol"].astype(str).str.strip().str.upper()
panel["issuer_nm"] = panel["issuer_nm"].astype(str).str.strip()
panel["entity_name"] = panel["entity_name"].astype(str).str.strip()
panel["sector"] = panel["sector"].astype(str).str.strip()
panel["grade"] = panel["grade"].astype(str).str.strip().str.upper()

numeric_cols = [
    "median_price", "mean_price", "median_yield", "mean_yield",
    "n_trades", "total_quantity", "vw_price", "vw_yield",
    "years_to_maturity_from_first_trace", "years_to_maturity_asof_2024_01_01"
]
for col in numeric_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel = panel.dropna(subset=["cusip_id", "trade_date", "mtrty_dt", "median_yield"]).copy()

# =========================================================
# LOAD + COMBINE TREASURY FILES
# =========================================================
t24 = load_treasury_file(treasury_2024_file)
t25 = load_treasury_file(treasury_2025_file)

treasury = (
    pd.concat([t24, t25], ignore_index=True)
    .sort_values("Date")
    .drop_duplicates(subset=["Date"])
    .reset_index(drop=True)
)

tenor_cols = [c for c in treasury.columns if c != "Date"]
tenor_map = {c: tenor_to_years(c) for c in tenor_cols}
tenor_cols = [c for c in tenor_cols if tenor_map[c] is not None]
tenor_years = np.array([tenor_map[c] for c in tenor_cols], dtype=float)

order = np.argsort(tenor_years)
tenor_cols = [tenor_cols[i] for i in order]
tenor_years = tenor_years[order]

treasury.to_csv(os.path.join(output_folder, "combined_treasury_curve_2024_2025.csv"), index=False)

# =========================================================
# MERGE TREASURY ON TRADE DATE
# =========================================================
spread_panel = panel.merge(
    treasury,
    left_on="trade_date",
    right_on="Date",
    how="left"
)

spread_panel = spread_panel.drop(columns=["Date"], errors="ignore")

# =========================================================
# REMAINING MATURITY
# =========================================================
spread_panel["years_to_maturity_on_trade_date"] = (
    (spread_panel["mtrty_dt"] - spread_panel["trade_date"]).dt.days / 365.25
)

spread_panel = spread_panel[spread_panel["years_to_maturity_on_trade_date"] > 0].copy()

# =========================================================
# PCHIP INTERPOLATION
# =========================================================
spread_panel["interp_treasury_yield_pchip"] = spread_panel.apply(
    lambda row: interpolate_treasury_yield_pchip(
        row=row,
        tenor_cols=tenor_cols,
        tenor_years=tenor_years,
        maturity_years=row["years_to_maturity_on_trade_date"]
    ),
    axis=1
)

# =========================================================
# SPREADS
# =========================================================
spread_panel["daily_spread_pchip"] = spread_panel["median_yield"] - spread_panel["interp_treasury_yield_pchip"]

if "mean_yield" in spread_panel.columns:
    spread_panel["daily_spread_mean_yield_pchip"] = spread_panel["mean_yield"] - spread_panel["interp_treasury_yield_pchip"]
if "vw_yield" in spread_panel.columns:
    spread_panel["daily_spread_vw_yield_pchip"] = spread_panel["vw_yield"] - spread_panel["interp_treasury_yield_pchip"]

spread_panel = spread_panel.dropna(subset=["interp_treasury_yield_pchip", "daily_spread_pchip"]).copy()

# =========================================================
# SAVE LONG PANEL
# =========================================================
spread_panel_out = spread_panel[
    [
        "cusip_id",
        "company_symbol",
        "issuer_nm",
        "entity_name",
        "sector",
        "grade",
        "trade_date",
        "mtrty_dt",
        "first_trace_date",
        "years_to_maturity_from_first_trace",
        "years_to_maturity_asof_2024_01_01",
        "years_to_maturity_on_trade_date",
        "median_price",
        "mean_price",
        "median_yield",
        "mean_yield",
        "vw_yield",
        "n_trades",
        "total_quantity",
        "interp_treasury_yield_pchip",
        "daily_spread_pchip",
        "daily_spread_mean_yield_pchip",
        "daily_spread_vw_yield_pchip",
    ]
].copy()

spread_panel_out.to_csv(os.path.join(output_folder, "cusip_day_spread_panel_pchip.csv"), index=False)

# =========================================================
# WIDE PANEL
# =========================================================
spread_panel_wide = spread_panel_out.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="daily_spread_pchip",
    aggfunc="first"
).sort_index()
spread_panel_wide.to_csv(os.path.join(output_folder, "cusip_day_spread_panel_pchip_wide.csv"))

# =========================================================
# SUMMARY
# =========================================================
spread_summary = (
    spread_panel_out.groupby("cusip_id", as_index=False)
    .agg(
        company_symbol=("company_symbol", "first"),
        issuer_name=("issuer_nm", "first"),
        entity_name=("entity_name", "first"),
        sector=("sector", "first"),
        grade=("grade", "first"),
        first_trade_date=("trade_date", "min"),
        last_trade_date=("trade_date", "max"),
        n_trade_days=("trade_date", "nunique"),
        avg_trades_per_day=("n_trades", "mean"),
        total_trades=("n_trades", "sum"),
        mean_spread=("daily_spread_pchip", "mean"),
        median_spread=("daily_spread_pchip", "median"),
        std_spread=("daily_spread_pchip", "std"),
        min_spread=("daily_spread_pchip", "min"),
        max_spread=("daily_spread_pchip", "max"),
        pct_negative_spread=("daily_spread_pchip", lambda x: np.mean(x < 0) if len(x) > 0 else np.nan),
        avg_remaining_maturity=("years_to_maturity_on_trade_date", "mean"),
        min_remaining_maturity=("years_to_maturity_on_trade_date", "min"),
        max_remaining_maturity=("years_to_maturity_on_trade_date", "max"),
        maturity_date=("mtrty_dt", "first"),
    )
    .sort_values(["n_trade_days", "total_trades"], ascending=[False, False])
    .reset_index(drop=True)
)
spread_summary.to_csv(os.path.join(output_folder, "cusip_day_spread_summary_pchip.csv"), index=False)

# =========================================================
# OVERLAP
# =========================================================
presence = spread_panel_out[["trade_date", "cusip_id"]].drop_duplicates().copy()
presence["present"] = 1

presence_wide = presence.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="present",
    aggfunc="max"
).fillna(0).astype(int)

cusips = list(presence_wide.columns)
overlap_rows = []

for i, c1 in enumerate(cusips):
    s1 = presence_wide[c1].values
    for c2 in cusips[i+1:]:
        s2 = presence_wide[c2].values
        overlap_days = int((s1 & s2).sum())
        overlap_rows.append({
            "cusip_1": c1,
            "cusip_2": c2,
            "overlap_days": overlap_days
        })

spread_pair_overlap = pd.DataFrame(overlap_rows).sort_values(
    ["overlap_days", "cusip_1", "cusip_2"],
    ascending=[False, True, True]
).reset_index(drop=True)

spread_pair_overlap.to_csv(os.path.join(output_folder, "cusip_day_spread_pair_overlap_pchip.csv"), index=False)

# =========================================================
# CONSTRUCTION REPORT
# =========================================================
report = pd.DataFrame({
    "metric": [
        "input_cusip_day_rows",
        "output_spread_rows",
        "unique_cusips",
        "unique_companies",
        "start_date",
        "end_date",
        "min_remaining_maturity_on_trade_date",
        "max_remaining_maturity_on_trade_date",
        "mean_daily_spread_pchip",
        "median_daily_spread_pchip"
    ],
    "value": [
        len(panel),
        len(spread_panel_out),
        spread_panel_out["cusip_id"].nunique(),
        spread_panel_out["company_symbol"].nunique(),
        spread_panel_out["trade_date"].min(),
        spread_panel_out["trade_date"].max(),
        spread_panel_out["years_to_maturity_on_trade_date"].min(),
        spread_panel_out["years_to_maturity_on_trade_date"].max(),
        spread_panel_out["daily_spread_pchip"].mean(),
        spread_panel_out["daily_spread_pchip"].median()
    ]
})
report.to_csv(os.path.join(output_folder, "spread_construction_report_pchip.csv"), index=False)

# =========================================================
# OPTIONAL COMPARISON AGAINST EXISTING LINEAR SPREAD PANEL
# =========================================================
if os.path.exists("cusip_day_spread_panel.csv"):
    linear_panel = pd.read_csv("cusip_day_spread_panel.csv", low_memory=False)
    linear_panel["trade_date"] = pd.to_datetime(linear_panel["trade_date"], errors="coerce")

    compare = linear_panel[["cusip_id", "trade_date", "daily_spread"]].merge(
        spread_panel_out[["cusip_id", "trade_date", "daily_spread_pchip"]],
        on=["cusip_id", "trade_date"],
        how="inner"
    )

    compare["spread_diff_pchip_minus_linear"] = compare["daily_spread_pchip"] - compare["daily_spread"]

    compare_summary = pd.DataFrame({
        "metric": [
            "matched_rows",
            "mean_abs_diff",
            "median_abs_diff",
            "max_abs_diff",
            "mean_diff"
        ],
        "value": [
            len(compare),
            compare["spread_diff_pchip_minus_linear"].abs().mean(),
            compare["spread_diff_pchip_minus_linear"].abs().median(),
            compare["spread_diff_pchip_minus_linear"].abs().max(),
            compare["spread_diff_pchip_minus_linear"].mean()
        ]
    })

    compare.to_csv(os.path.join(output_folder, "pchip_vs_linear_row_level_comparison.csv"), index=False)
    compare_summary.to_csv(os.path.join(output_folder, "pchip_vs_linear_summary.csv"), index=False)

# =========================================================
# ZIP OUTPUTS
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== PCHIP SPREAD CONSTRUCTION COMPLETE ===")
print(f"Input CUSIP-day rows: {len(panel):,}")
print(f"Output spread rows: {len(spread_panel_out):,}")
print(f"Unique CUSIPs: {spread_panel_out['cusip_id'].nunique():,}")
print(f"Unique companies: {spread_panel_out['company_symbol'].nunique():,}")
print(f"Date range: {spread_panel_out['trade_date'].min()} to {spread_panel_out['trade_date'].max()}")
print(f"Mean daily spread (PCHIP): {spread_panel_out['daily_spread_pchip'].mean():.6f}")
print(f"Median daily spread (PCHIP): {spread_panel_out['daily_spread_pchip'].median():.6f}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== PCHIP SPREAD CONSTRUCTION COMPLETE ===
Input CUSIP-day rows: 919,129
Output spread rows: 916,832
Unique CUSIPs: 2,331
Unique companies: 229
Date range: 2024-01-02 00:00:00 to 2025-12-31 00:00:00
Mean daily spread (PCHIP): 0.679524
Median daily spread (PCHIP): 0.650805
Output folder created: spread_construction_outputs_pchip_2024_2025
Zip file created: spread_construction_outputs_pchip_2024_2025.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

# =========================================================
# FILE PATHS
# =========================================================
cleaned_panel_file = "cusip_day_spread_panel_cleaned.csv"
cleaned_summary_file = "cusip_day_spread_summary_cleaned.csv"
cleaned_overlap_file = "cusip_day_spread_pair_overlap_cleaned.csv"

output_folder = "pair_universe_outputs_cross_sector"
zip_filename = "pair_universe_outputs_cross_sector.zip"

os.makedirs(output_folder, exist_ok=True)

# =========================================================
# PARAMETERS
# These are conservative defaults and can be adjusted later.
# =========================================================
min_overlap_days = 150
max_avg_remaining_maturity_diff = 2.0
min_abs_spread_change_corr = 0.20
min_trade_days_per_bond = 150
min_total_trades_per_bond = 300

# =========================================================
# LOAD FILES
# =========================================================
panel = pd.read_csv(cleaned_panel_file, low_memory=False)
summary = pd.read_csv(cleaned_summary_file, low_memory=False)
overlap = pd.read_csv(cleaned_overlap_file, low_memory=False)

# =========================================================
# STANDARDIZE TYPES
# =========================================================
date_cols_panel = ["trade_date", "mtrty_dt", "first_trace_date"]
for col in date_cols_panel:
    if col in panel.columns:
        panel[col] = pd.to_datetime(panel[col], errors="coerce")

date_cols_summary = ["first_trade_date", "last_trade_date", "maturity_date"]
for col in date_cols_summary:
    if col in summary.columns:
        summary[col] = pd.to_datetime(summary[col], errors="coerce")

str_cols_panel = ["cusip_id", "company_symbol", "issuer_nm", "entity_name", "sector", "grade"]
for col in str_cols_panel:
    if col in panel.columns:
        panel[col] = panel[col].astype(str).str.strip()

str_cols_summary = ["cusip_id", "company_symbol", "issuer_name", "entity_name", "sector", "grade"]
for col in str_cols_summary:
    if col in summary.columns:
        summary[col] = summary[col].astype(str).str.strip()

numeric_cols_panel = [
    "daily_spread",
    "n_trades",
    "total_quantity",
    "years_to_maturity_on_trade_date"
]
for col in numeric_cols_panel:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

numeric_cols_summary = [
    "n_trade_days",
    "avg_trades_per_day",
    "total_trades",
    "mean_spread",
    "median_spread",
    "std_spread",
    "pct_negative_spread",
    "avg_remaining_maturity",
    "min_remaining_maturity",
    "max_remaining_maturity",
]
for col in numeric_cols_summary:
    if col in summary.columns:
        summary[col] = pd.to_numeric(summary[col], errors="coerce")

overlap["cusip_1"] = overlap["cusip_1"].astype(str).str.strip()
overlap["cusip_2"] = overlap["cusip_2"].astype(str).str.strip()
overlap["overlap_days"] = pd.to_numeric(overlap["overlap_days"], errors="coerce")

# =========================================================
# BOND-LEVEL METADATA
# =========================================================
bond_meta = summary[[
    "cusip_id",
    "company_symbol",
    "issuer_name",
    "entity_name",
    "sector",
    "grade",
    "n_trade_days",
    "avg_trades_per_day",
    "total_trades",
    "mean_spread",
    "median_spread",
    "std_spread",
    "pct_negative_spread",
    "avg_remaining_maturity",
    "min_remaining_maturity",
    "max_remaining_maturity",
]].copy()

bond_meta = bond_meta.drop_duplicates(subset=["cusip_id"]).reset_index(drop=True)

meta_1 = bond_meta.rename(columns={
    "cusip_id": "cusip_1",
    "company_symbol": "company_symbol_1",
    "issuer_name": "issuer_name_1",
    "entity_name": "entity_name_1",
    "sector": "sector_1",
    "grade": "grade_1",
    "n_trade_days": "n_trade_days_1",
    "avg_trades_per_day": "avg_trades_per_day_1",
    "total_trades": "total_trades_1",
    "mean_spread": "mean_spread_1",
    "median_spread": "median_spread_1",
    "std_spread": "std_spread_1",
    "pct_negative_spread": "pct_negative_spread_1",
    "avg_remaining_maturity": "avg_remaining_maturity_1",
    "min_remaining_maturity": "min_remaining_maturity_1",
    "max_remaining_maturity": "max_remaining_maturity_1",
})

meta_2 = bond_meta.rename(columns={
    "cusip_id": "cusip_2",
    "company_symbol": "company_symbol_2",
    "issuer_name": "issuer_name_2",
    "entity_name": "entity_name_2",
    "sector": "sector_2",
    "grade": "grade_2",
    "n_trade_days": "n_trade_days_2",
    "avg_trades_per_day": "avg_trades_per_day_2",
    "total_trades": "total_trades_2",
    "mean_spread": "mean_spread_2",
    "median_spread": "median_spread_2",
    "std_spread": "std_spread_2",
    "pct_negative_spread": "pct_negative_spread_2",
    "avg_remaining_maturity": "avg_remaining_maturity_2",
    "min_remaining_maturity": "min_remaining_maturity_2",
    "max_remaining_maturity": "max_remaining_maturity_2",
})

pairs = overlap.merge(meta_1, on="cusip_1", how="left").merge(meta_2, on="cusip_2", how="left")

# =========================================================
# STRICT CROSS-SECTOR FILTER
# Your instruction: ONLY pairs from different sectors
# =========================================================
pairs["cross_sector"] = (pairs["sector_1"] != pairs["sector_2"]).astype(int)
pairs["same_company"] = (pairs["company_symbol_1"] == pairs["company_symbol_2"]).astype(int)

pairs = pairs[
    (pairs["cross_sector"] == 1) &
    (pairs["same_company"] == 0)
].copy()

# Canonical sector-combo label for grouping
pairs["sector_combo"] = pairs.apply(
    lambda r: " | ".join(sorted([str(r["sector_1"]), str(r["sector_2"])])),
    axis=1
)

# =========================================================
# PAIR DIAGNOSTICS
# =========================================================
pairs["min_trade_days_across_bonds"] = pairs[["n_trade_days_1", "n_trade_days_2"]].min(axis=1)
pairs["min_total_trades_across_bonds"] = pairs[["total_trades_1", "total_trades_2"]].min(axis=1)
pairs["avg_remaining_maturity_diff"] = (
    pairs["avg_remaining_maturity_1"] - pairs["avg_remaining_maturity_2"]
).abs()
pairs["mean_spread_diff"] = (
    pairs["mean_spread_1"] - pairs["mean_spread_2"]
).abs()
pairs["std_spread_diff"] = (
    pairs["std_spread_1"] - pairs["std_spread_2"]
).abs()

pairs["overlap_share_of_shorter_bond"] = pairs["overlap_days"] / pairs["min_trade_days_across_bonds"]

# =========================================================
# SPREAD-CHANGE CORRELATION
# Compute from cleaned panel using aligned daily spread changes
# =========================================================
spread_wide = panel.pivot_table(
    index="trade_date",
    columns="cusip_id",
    values="daily_spread",
    aggfunc="first"
).sort_index()

spread_change_wide = spread_wide.diff()

corr_matrix = spread_change_wide.corr(min_periods=60)
corr_cols = list(corr_matrix.columns)
corr_values = corr_matrix.to_numpy()

cusip_to_idx = {c: i for i, c in enumerate(corr_cols)}

idx1 = pairs["cusip_1"].map(cusip_to_idx).fillna(-1).astype(int).to_numpy()
idx2 = pairs["cusip_2"].map(cusip_to_idx).fillna(-1).astype(int).to_numpy()

spread_change_corr = np.full(len(pairs), np.nan)
valid_idx = (idx1 >= 0) & (idx2 >= 0)
spread_change_corr[valid_idx] = corr_values[idx1[valid_idx], idx2[valid_idx]]

pairs["spread_change_corr"] = spread_change_corr
pairs["abs_spread_change_corr"] = pairs["spread_change_corr"].abs()

# =========================================================
# SAVE FULL CROSS-SECTOR CANDIDATE PAIR UNIVERSE
# =========================================================
candidate_pair_file = os.path.join(output_folder, "candidate_pair_universe_cross_sector.csv")
pairs.sort_values(
    ["overlap_days", "abs_spread_change_corr", "avg_remaining_maturity_diff"],
    ascending=[False, False, True]
).to_csv(candidate_pair_file, index=False)

# =========================================================
# SHORTLIST FOR MODELING
# Conservative shortlist for Johansen/VECM input
# =========================================================
pair_shortlist = pairs[
    (pairs["overlap_days"] >= min_overlap_days) &
    (pairs["n_trade_days_1"] >= min_trade_days_per_bond) &
    (pairs["n_trade_days_2"] >= min_trade_days_per_bond) &
    (pairs["total_trades_1"] >= min_total_trades_per_bond) &
    (pairs["total_trades_2"] >= min_total_trades_per_bond) &
    (pairs["avg_remaining_maturity_diff"] <= max_avg_remaining_maturity_diff) &
    (pairs["abs_spread_change_corr"] >= min_abs_spread_change_corr)
].copy()

pair_shortlist = pair_shortlist.sort_values(
    ["overlap_days", "abs_spread_change_corr", "avg_remaining_maturity_diff"],
    ascending=[False, False, True]
).reset_index(drop=True)

shortlist_file = os.path.join(output_folder, "pair_universe_shortlist_cross_sector.csv")
pair_shortlist.to_csv(shortlist_file, index=False)

# =========================================================
# FINAL MODEL BOND UNIVERSE
# Bonds that appear in at least one shortlisted cross-sector pair
# =========================================================
final_bond_ids = pd.unique(
    pd.concat([
        pair_shortlist["cusip_1"],
        pair_shortlist["cusip_2"]
    ], ignore_index=True)
)

final_model_bond_universe = bond_meta[bond_meta["cusip_id"].isin(final_bond_ids)].copy()
final_model_bond_universe = final_model_bond_universe.sort_values(
    ["n_trade_days", "total_trades"],
    ascending=[False, False]
).reset_index(drop=True)

final_bond_file = os.path.join(output_folder, "final_model_bond_universe_cross_sector.csv")
final_model_bond_universe.to_csv(final_bond_file, index=False)

# =========================================================
# SECTOR-COMBO COUNTS
# =========================================================
sector_combo_counts = (
    pair_shortlist.groupby("sector_combo", as_index=False)
    .agg(
        n_pairs=("sector_combo", "size"),
        mean_overlap_days=("overlap_days", "mean"),
        median_overlap_days=("overlap_days", "median"),
        mean_abs_spread_change_corr=("abs_spread_change_corr", "mean"),
        median_maturity_diff=("avg_remaining_maturity_diff", "median")
    )
    .sort_values(["n_pairs", "mean_abs_spread_change_corr"], ascending=[False, False])
    .reset_index(drop=True)
)

sector_combo_file = os.path.join(output_folder, "cross_sector_pair_counts_by_sector_combo.csv")
sector_combo_counts.to_csv(sector_combo_file, index=False)

# =========================================================
# FILTER REPORT
# =========================================================
filter_report = pd.DataFrame({
    "metric": [
        "input_cleaned_pairs",
        "cross_sector_pairs_only",
        "shortlisted_pairs",
        "final_model_bonds_in_shortlist",
        "unique_sector_combos_in_shortlist",
        "min_overlap_days_threshold",
        "max_avg_remaining_maturity_diff_threshold",
        "min_abs_spread_change_corr_threshold",
        "min_trade_days_per_bond_threshold",
        "min_total_trades_per_bond_threshold"
    ],
    "value": [
        len(overlap),
        len(pairs),
        len(pair_shortlist),
        len(final_model_bond_universe),
        sector_combo_counts["sector_combo"].nunique(),
        min_overlap_days,
        max_avg_remaining_maturity_diff,
        min_abs_spread_change_corr,
        min_trade_days_per_bond,
        min_total_trades_per_bond
    ]
})

filter_report_file = os.path.join(output_folder, "pair_universe_filter_report_cross_sector.csv")
filter_report.to_csv(filter_report_file, index=False)

# =========================================================
# ZIP OUTPUTS
# =========================================================
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_folder):
        full_path = os.path.join(output_folder, fname)
        zf.write(full_path, arcname=fname)

# =========================================================
# PRINT CHECKS
# =========================================================
print("=== CROSS-SECTOR PAIR UNIVERSE CONSTRUCTION COMPLETE ===")
print(f"Input cleaned pairs: {len(overlap):,}")
print(f"Cross-sector-only candidate pairs: {len(pairs):,}")
print(f"Shortlisted cross-sector pairs: {len(pair_shortlist):,}")
print(f"Final model bonds appearing in shortlist: {len(final_model_bond_universe):,}")
print(f"Unique sector combos in shortlist: {sector_combo_counts['sector_combo'].nunique():,}")
print(f"Output folder created: {output_folder}")
print(f"Zip file created: {zip_filename}")

try:
    from google.colab import files
    files.download(zip_filename)
except Exception:
    print("Not running in Google Colab; automatic download skipped.")

=== CROSS-SECTOR PAIR UNIVERSE CONSTRUCTION COMPLETE ===
Input cleaned pairs: 2,636,956
Cross-sector-only candidate pairs: 2,352,831
Shortlisted cross-sector pairs: 73,777
Final model bonds appearing in shortlist: 663
Unique sector combos in shortlist: 55
Output folder created: pair_universe_outputs_cross_sector
Zip file created: pair_universe_outputs_cross_sector.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>